# NCTB bilingual grade-constrained RAG — reproducible audit v2

**One notebook • Kaggle T4 (one GPU or T4×2) • resumable stages • no paid APIs**

### Start here
1. Attach `codderboy/nctb-book-final-v3` and `codderboy/nctb-valid-qa-pair`.
2. Enable **GPU T4** and **Internet**. Start a fresh kernel and **Run All**.
3. Outputs: `/kaggle/working/nctb_grade_audit_v2/`. Use **Save Version → Save & Run All** to persist Kaggle output. Download/retain completed versions.
4. To resume, attach a previous notebook output or a dataset containing that output folder. Automatic discovery accepts one matching archive; set `RESUME_FROM` explicitly if several exist. Run this same notebook again.
5. Add optional `HF_TOKEN` in Kaggle Secrets for gated models (after accepting their licenses). No token is needed for the default Qwen/BGE models.

**Scientific scope.** This is a corrected *exploratory* study on an already inspected QA pool. It does not manufacture a new confirmatory test set, teacher ratings, relevance judgments or learning outcomes. Default reranker supervision is explicitly labeled **weak supervision**; judged candidate pairs can replace those labels. Default splits hold out book families and detected near-duplicate source groups. Original textbook PDFs and manual bilingual alignment are needed to fully certify corpus provenance.

**Time budget.** A session stops admitting GPU batches after 10.75 hours, leaving approximately 75 minutes before the requested 12-hour ceiling for scoring and finalization. There is no guarantee every optional block fits. Completed records resume; nothing silently changes to a smaller test set or a different system. In-flight uncommitted work can be lost on abrupt termination, and local checkpoints are not a substitute for a durable saved Kaggle version.

**Core changes:** native chat templates; question-preserving token packing; explicit answer language; full delivered prompts/token IDs; strict grade filtering without backfill; fixed-prompt comparisons plus a prompt factorial; correct E5 prefixes; grade-input/no-grade-input reranker ablations; grouped splits; development-only tuning; per-question results, failure statuses and cluster-aware uncertainty; teacher packet/export/import.


In [1]:
# CELL 1 (ONE-SHOT, Save-Version-safe) — never touches numpy/scipy/pyarrow/pandas.
# Kaggle's base image ships these as a coherent, ABI-matched set alongside scikit-learn,
# xgboost, etc. Reinstalling them (even "cleanly") caused ImportError before, because
# Kaggle's overlay filesystem can leave stale compiled (.so) fragments behind. So we
# freeze them at whatever version is already installed and never let pip touch them.
import sys, subprocess, time, importlib.metadata as metadata
if 'NOTEBOOK_START' not in globals(): NOTEBOOK_START = time.monotonic()

def pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args],
                           check=check, capture_output=True, text=True)

FROZEN = ['numpy', 'scipy', 'pyarrow', 'pandas', 'matplotlib']
freeze_constraints = []
for name in FROZEN:
    try:
        freeze_constraints.append(f'{name}=={metadata.version(name)}')
    except metadata.PackageNotFoundError:
        raise RuntimeError(f'{name} is expected to be preinstalled by the Kaggle image but is missing.')

# Packages we actually need for this project, installed WITH the freeze constraints
# passed alongside them, so pip's resolver cannot silently upgrade numpy/scipy/etc.
PINS = {
    'huggingface-hub': '0.36.0', 'tokenizers': '0.22.2', 'transformers': '4.57.6',
    'accelerate': '1.12.0', 'sentence-transformers': '5.2.0', 'bitsandbytes': '0.49.2',
    'sentencepiece': '0.2.0', 'safetensors': '0.7.0', 'jinja2': '3.1.6',
    'rank-bm25': '0.2.2', 'sacrebleu': '2.5.1', 'rouge-score': '0.1.2',
    'datasketch': '1.6.5', 'krippendorff': '0.8.2',
}
need = []
for name, ver in PINS.items():
    try:
        cur = metadata.version(name)
    except metadata.PackageNotFoundError:
        cur = None
    if cur != ver:
        need.append(f'{name}=={ver}')

if need:
    result = pip('install', '--quiet', '--disable-pip-version-check',
                 *need, *freeze_constraints, check=False)
    if result.returncode != 0:
        print(result.stdout[-4000:]); print(result.stderr[-4000:])
        raise RuntimeError('pip install failed; see output above.')

# faiss-cpu only needs numpy at runtime (already satisfied); install with --no-deps so
# its own resolver never gets a chance to touch numpy/scipy.
try:
    cur = metadata.version('faiss-cpu')
except metadata.PackageNotFoundError:
    cur = None
if cur != '1.13.2':
    r = pip('install', '--quiet', '--disable-pip-version-check', '--no-deps', 'faiss-cpu==1.13.2', check=False)
    if r.returncode != 0:
        print(r.stdout[-4000:]); print(r.stderr[-4000:])
        raise RuntimeError('faiss-cpu install failed; see output above.')

# Record final resolved versions for later environment logging (Cell 3 reads this dict
# instead of assuming a fixed PINS set, since numpy/scipy/pyarrow/pandas/matplotlib are
# whatever Kaggle shipped, not something we pinned ourselves).
ALL_TRACKED_PACKAGES = {**{n: metadata.version(n) for n in FROZEN}, **PINS, 'faiss-cpu': '1.13.2'}

print('Environment ready in one pass (numpy/scipy/pandas/pyarrow/matplotlib untouched).')
print({k: ALL_TRACKED_PACKAGES[k] for k in FROZEN})

Environment ready in one pass (numpy/scipy/pandas/pyarrow/matplotlib untouched).
{'numpy': '2.0.2', 'scipy': '1.16.3', 'pyarrow': '24.0.0', 'pandas': '2.3.3', 'matplotlib': '3.10.0'}


In [2]:
# CELL 2 — edit these settings; defaults target one T4 and also work on T4×2.
from pathlib import Path
CORPUS_ROOT = Path('/kaggle/input/datasets/codderboy/nctb-book-final-v3')
QA_ROOT = Path('/kaggle/input/datasets/codderboy/nctb-valid-qa-pair')
ROOT = Path('/kaggle/working/nctb_grade_audit_v2')
RESUME_FROM = None  # e.g. '/kaggle/input/my-previous-run/nctb_grade_audit_v2'
AUTO_RESUME = True

CFG = dict(
    schema='nctb-grade-audit-v2.1', seed=42, n_test=200, n_dev=100,
    split_unit='book', near_duplicate_threshold=0.85,
    final_k=5, candidate_k=60, search_depth=120,
    embedding_batch=24, embedding_shard=512, embedding_max_tokens=512,
    run_e5=True, run_rerankers=True, ce_base='microsoft/Multilingual-MiniLM-L12-H384',
    ce_seeds=[42,43,44], ce_variants=['explicit_grade','no_grade_input'],
    ce_epochs=3, ce_batch=12, ce_max_tokens=384, ce_lr=2e-5,
    ce_checkpoint_steps=50, ce_candidates_train=8, ce_candidates_eval=60,
    generation_models=['qwen2.5-7b'],  # add 'phi3.5-mini', 'llama3.1-8b', 'gemma2-9b'
    gen_batch=4, gen_input_tokens=2048, gen_output_tokens=128,
    gen_passage_tokens=320, generation_device=0, max_attempts=2,
    gpu_work_hours=10.75, min_stage_reserve_minutes=15,
    bootstrap_replicates=2000, alpha_grid=[0.,0.05,0.1,0.15,0.25,0.4,0.8],
    utility_gcr_weight=0.25, cmr_threshold=0.7, primary_seed=42,
    teacher_questions=80, teacher_raters=['rater_1','rater_2'],
)
# Optional genuine human inputs: leave None until these files actually exist.
QA_REVIEW_CSV = None        # columns: qa_id, decision [keep/exclude], corrected_question, corrected_answer, answer_language
RELEVANCE_CSV = None        # columns: qa_id, chunk_id, relevance [0..1]; TRAIN groups only
BOOK_ALIGNMENT_CSV = None  # columns: book_id, family_id (independent manual bilingual/edition alignment)
TEACHER_RATINGS_CSV = None  # completed blind annotation CSV exported later in this notebook
MODEL_REGISTRY = {
 'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
 'phi3.5-mini': 'microsoft/Phi-3.5-mini-instruct',
 'llama3.1-8b': 'meta-llama/Llama-3.1-8B-Instruct',
 'gemma2-9b': 'google/gemma-2-9b-it',
 'qwen2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
}
EMBEDDERS={'bge_m3':'BAAI/bge-m3','e5':'intfloat/multilingual-e5-large'}
RELEVANCE_MODEL='BAAI/bge-reranker-v2-m3'


In [3]:
# CELL 3 — artifact safety, session budget, environment and model revisions
import os, json, time, math, random, hashlib, shutil, gc, re, unicodedata, traceback, platform
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import wilcoxon
from huggingface_hub import HfApi
SESSION_START=globals().get('NOTEBOOK_START',time.monotonic())
class BudgetStop(Exception): pass
class Budget:
    def left(self): return CFG['gpu_work_hours']*3600-(time.monotonic()-SESSION_START)
    def check(self, seconds=60):
        if self.left()<seconds: raise BudgetStop('Session GPU deadline reached; resume pending work in another session.')
BUDGET=Budget()

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str,separators=(',',':')).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    with open(temp,'w',encoding='utf-8') as f:
        json.dump(obj,f,ensure_ascii=False,indent=2,default=lambda x: x.item() if isinstance(x,np.generic) else str(x));f.flush();os.fsync(f.fileno())
    os.replace(temp,path)
def atomic_parquet(path,df):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    df.to_parquet(tmp,index=False);os.replace(tmp,path)
def atomic_npy(path,array):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    with open(tmp,'wb') as f: np.save(f,array);f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)
def read_json(path,default=None):
    return json.loads(Path(path).read_text()) if Path(path).exists() else default

def restore():
    marker=ROOT/'artifact_manifest.json'
    if marker.exists():
        if read_json(marker)['schema']!=CFG['schema']: raise RuntimeError('Output schema mismatch. Choose a new ROOT.')
        return
    source=Path(RESUME_FROM) if RESUME_FROM else None
    if source is None and AUTO_RESUME and Path('/kaggle/input').exists():
        found=[]
        for p in Path('/kaggle/input').rglob('artifact_manifest.json'):
            try:
                if read_json(p).get('schema')==CFG['schema']:found.append(p.parent)
            except Exception: pass
        if len(found)>1: raise RuntimeError('Multiple resume archives found. Set RESUME_FROM explicitly: '+str(found))
        if found:source=found[0]
    ROOT.mkdir(parents=True,exist_ok=True)
    if source:
        if read_json(source/'artifact_manifest.json',{}).get('schema')!=CFG['schema']:raise ValueError('Incompatible resume folder')
        print('Restoring durable archive:',source,flush=True)
        shutil.copytree(source,ROOT,dirs_exist_ok=True)
    atomic_json(marker,{'schema':CFG['schema'],'created_utc':datetime.now(timezone.utc).isoformat()})
restore()
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOG_PATH=ROOT/'logs'/f'{RUN_ID}.jsonl';LOG_PATH.parent.mkdir(exist_ok=True)
def log(stage,message,**extra):
    row=dict(time=datetime.now(timezone.utc).isoformat(),stage=stage,message=message,
             elapsed_min=round((time.monotonic()-SESSION_START)/60,2),remaining_min=round(max(0,BUDGET.left())/60,2),**extra)
    print(f"[{row['elapsed_min']:7.1f} min | {row['remaining_min']:6.1f} min left] {stage}: {message}",flush=True)
    with open(LOG_PATH,'a') as f:f.write(json.dumps(row,default=str)+'\n')
def stage_state(stage,status,**details):
    p=ROOT/'stage_status.json';o=read_json(p,{})
    o[stage]=dict(status=status,updated=datetime.now(timezone.utc).isoformat(),**details);atomic_json(p,o)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
os.environ['TOKENIZERS_PARALLELISM']='false'
torch.backends.cudnn.benchmark=False
if not torch.cuda.is_available():raise RuntimeError('Enable Kaggle GPU T4 and restart. CPU-only generation is deliberately disabled.')
DEVICE=f"cuda:{CFG['generation_device']}"
if CFG['generation_device']>=torch.cuda.device_count():raise ValueError('Configured CUDA device is absent')
def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
HF_TOKEN=os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=HF_TOKEN or UserSecretsClient().get_secret('HF_TOKEN')
except Exception:pass
REV_PATH=ROOT/'model_revisions.json';REVISIONS=read_json(REV_PATH,{})
def revision(repo):
    if repo not in REVISIONS:
        REVISIONS[repo]=HfApi(token=HF_TOKEN).model_info(repo).sha
        atomic_json(REV_PATH,REVISIONS)
    return REVISIONS[repo]
ENV={'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,
     'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
     'packages':{k:metadata.version(k) for k in PINS},'config':CFG}
atomic_json(ROOT/'logs'/f'{RUN_ID}_environment.json',ENV)
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(ROOT/'logs'/f'{RUN_ID}_pip_freeze.txt','w'),check=True)
log('setup','GPU available; use one device at a time to avoid implicit CPU offload.',gpus=ENV['gpus'])

def tokens(text):return re.findall(r'[ঀ-৿]+|[A-Za-z0-9]+',unicodedata.normalize('NFC',str(text)).lower())
def norm(text):return ' '.join(tokens(text))
def script_language(text):
    bn=len(re.findall(r'[ঀ-৿]',str(text)));en=len(re.findall('[A-Za-z]',str(text)))
    return 'bn' if bn>en else 'en'


[    0.6 min |  644.5 min left] setup: GPU available; use one device at a time to avoid implicit CPU offload.


## Data validation and book-grouped splits

The default test set is a **new exploratory split of the supplied QA pool**, not the old 200-question sample. Entire held-out book families and detected source-duplicate groups are excluded from reranker training and lexical-vocabulary fitting; their passages remain in the retrieval index. Automatic alignment uses normalized subject/book labels and source similarity. Inspect `book_alignment_audit.csv`; supply a manual alignment map for certification.

Combined `9-10` and `11-12` labels remain bands. A passage is eligible when its **upper band endpoint ≤ the requested band's upper endpoint**. Report these as band-level comparisons; do not present them as exact grade-9 or grade-11 appropriateness. Teacher review overrides are versioned and invalidate dependent artifacts.


In [4]:
# CELL 4 — load, canonicalize, detect source duplicates, and freeze grouped partitions
from datasketch import MinHash, MinHashLSH

def locate(root,name):
    p=root/name
    if p.exists():return p
    hits=list(root.rglob(name)) if root.exists() else []
    if not hits: hits=list(Path('/kaggle/input').rglob(name))
    if len(hits)!=1:raise FileNotFoundError(f'Expected one {name}; found {hits}. Correct the configured dataset path.')
    return hits[0]
CORPUS_FILE=locate(CORPUS_ROOT,'chunks_clean.parquet')
QA_FILE=locate(QA_ROOT,'synthetic_qa_validated.jsonl')
source_hashes={'corpus':file_hash(CORPUS_FILE),'qa':file_hash(QA_FILE)}
for name,path in [('review',QA_REVIEW_CSV),('alignment',BOOK_ALIGNMENT_CSV)]:
    if path:source_hashes[name]=file_hash(path)
DATA_KEY=digest([CFG['schema'],source_hashes,CFG['seed'],CFG['split_unit'],CFG['n_test'],CFG['n_dev'],CFG['near_duplicate_threshold']])[:20]
STUDY=ROOT/'studies'/DATA_KEY;STUDY.mkdir(parents=True,exist_ok=True)
DATA=STUDY/'data';DATA.mkdir(exist_ok=True)
TAB=STUDY/'tables';FIG=STUDY/'figures';TAB.mkdir(exist_ok=True);FIG.mkdir(exist_ok=True)

def band(value):
    nums=re.findall(r'\d+',str(value))
    if not nums:raise ValueError('Unresolved grade: '+str(value))
    lo=int(nums[0]);hi=int(nums[1]) if len(nums)>1 else lo
    if not(1<=lo<=hi<=12):raise ValueError('Invalid grade band '+str(value))
    return lo,hi
raw=pd.read_parquet(CORPUS_FILE)
required={'chunk_id','text','class','book_id','subject','quality_tier'}
if not required.issubset(raw):raise ValueError('Missing corpus columns: '+str(required-set(raw)))
corpus=raw.loc[raw.quality_tier.isin(['clean','usable'])].copy().reset_index(drop=True)
if corpus.chunk_id.duplicated().any():raise ValueError('Nonunique chunk IDs; fix the corpus before retrieval.')
if corpus.text.isna().any() or corpus.text.str.strip().eq('').any():raise ValueError('Empty corpus text')
corpus['chunk_id']=corpus.chunk_id.astype(str)
corpus[['grade_min','grade_max']]=pd.DataFrame([band(x) for x in corpus['class']],index=corpus.index)
corpus['grade_label']=[str(a) if a==b else f'{a}-{b}' for a,b in zip(corpus.grade_min,corpus.grade_max)]
corpus['language']=corpus.get('lang_detected',corpus.get('language',pd.Series('unknown',index=corpus.index))).astype(str).str.lower().replace({'ben':'bn','eng':'en'})
corpus['pos']=np.arange(len(corpus))
# Align same subject/grade across language editions. Review optional mappings before publication.
corpus['book_family']=[f'{g}|{norm(s)}' for g,s in zip(corpus.grade_label,corpus.subject)]
if BOOK_ALIGNMENT_CSV:
    align=pd.read_csv(BOOK_ALIGNMENT_CSV,dtype=str)
    if align.book_id.duplicated().any():raise ValueError('Duplicate book IDs in alignment map')
    amap=dict(zip(align.book_id,align.family_id))
    corpus['book_family']=[amap.get(str(b),str(f)) for b,f in zip(corpus.book_id,corpus.book_family)]
corpus[['book_id','book_family','grade_label','subject','language']].drop_duplicates().to_csv(DATA/'book_alignment_audit.csv',index=False)
qa=pd.read_json(QA_FILE,lines=True)
qa['qa_id']=[digest([str(r.chunk_id),str(r.question),str(r.answer)])[:24] for r in qa.itertuples()]
if qa.qa_id.duplicated().any():raise ValueError('Duplicate QA identities require explicit deduplication')
qa['original_question']=qa.question;qa['original_answer']=qa.answer
qa['answer_language']=qa.question.map(script_language)
qa['review_status']='unreviewed'
if QA_REVIEW_CSV:
    review=pd.read_csv(QA_REVIEW_CSV).fillna('')
    if review.qa_id.duplicated().any():raise ValueError('Duplicate QA reviews')
    overrides=review.set_index('qa_id').to_dict('index')
    for i,r in qa.iterrows():
        x=overrides.get(r.qa_id,{})
        if x.get('decision') not in ['',None,'keep','exclude']:raise ValueError('Review decision must be keep/exclude')
        qa.at[i,'review_status']=x.get('decision') or 'unreviewed'
        for col,src in [('question','corrected_question'),('answer','corrected_answer'),('answer_language','answer_language')]:
            if x.get(src):qa.at[i,col]=x[src]
    qa=qa[qa.review_status!='exclude'].copy()
if not qa.answer_language.isin(['bn','en']).all():raise ValueError('Answer language must be bn or en')
qa=qa.dropna(subset=['question','answer']).copy()
posmap=dict(zip(corpus.chunk_id,corpus.pos));qa['gold_pos']=qa.chunk_id.astype(str).map(posmap)
unresolved=qa[qa.gold_pos.isna()].copy();atomic_parquet(DATA/'unresolved_qa.parquet',unresolved)
qa=qa[qa.gold_pos.notna()].reset_index(drop=True);qa['gold_pos']=qa.gold_pos.astype(int)
gold=corpus.iloc[qa.gold_pos].reset_index(drop=True)
qa['source_text_matches']=qa.source_text.astype(str).to_numpy()==gold.text.astype(str).to_numpy()
qa['source_text']=gold.text.to_numpy()  # canonical evidence, not a stale truncated copy
for col in ['grade_min','grade_max','grade_label','book_id','book_family','subject']:
    qa[col]=gold[col].to_numpy()
qa['answer_exact_in_source']=[str(a) in str(s) for a,s in zip(qa.answer,qa.source_text)]

class UnionFind:
    def __init__(self,items):self.p={x:x for x in items}
    def find(self,x):
        while self.p[x]!=x:self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b:self.p[max(a,b)]=min(a,b)
uf=UnionFind(corpus.book_family.unique().tolist())
# Source-text near duplicates among QA-bearing passages merge families, not just individual questions.
unique_gold=qa.drop_duplicates('gold_pos');lsh=MinHashLSH(threshold=CFG['near_duplicate_threshold'],num_perm=64)
seen={};shingles={};near_pairs=[]
for r in unique_gold.itertuples():
    txt=norm(r.source_text);sh={txt[j:j+5] for j in range(max(1,len(txt)-4))};mh=MinHash(num_perm=64,seed=42)
    mh.update_batch([x.encode() for x in sorted(sh)])
    for key in lsh.query(mh):
        sim=len(sh&shingles[key])/max(1,len(sh|shingles[key]))
        if sim>=CFG['near_duplicate_threshold']:
            uf.union(r.book_family,seen[key]);near_pairs.append({'left':r.chunk_id,'right':key,'jaccard':sim})
    key=str(r.chunk_id);lsh.insert(key,mh);seen[key]=r.book_family;shingles[key]=sh
# Exact repeated normalized questions must not cross splits either.
for _,g in qa.groupby(qa.question.map(norm)):
    fam=g.book_family.tolist()
    for b in fam[1:]:uf.union(fam[0],b)
corpus['group_id']=corpus.book_family.map(uf.find);qa['group_id']=qa.book_family.map(uf.find)
if CFG['split_unit']!='book':raise ValueError('This released notebook certifies book-group splits only; do not silently weaken them.')

def select_groups(frame,target,seed):
    rng=np.random.default_rng(seed);counts=frame.groupby(['group_id','grade_label','answer_language']).size()
    strata=sorted(set(zip(frame.grade_label,frame.answer_language)));desired=target/max(1,len(strata))
    available=list(frame.group_id.unique());rng.shuffle(available);chosen=[];current=Counter();n=0
    while n<target and available:
        def priority(g):
            part=counts.loc[g];gain=sum(min(v,max(0,desired-current[k])) for k,v in part.items())
            return gain/max(1,float(part.sum()))-0.01*abs(n+part.sum()-target)/max(1,target)
        best=max(available,key=priority);available.remove(best);chosen.append(best)
        part=counts.loc[best]
        for k,v in part.items():current[k]+=int(v)
        n+=int(part.sum())
    return chosen

def balanced_take(frame,n,seed):
    if n>=len(frame):return frame.copy()
    buckets=[g.sample(frac=1,random_state=seed).index.tolist() for _,g in frame.groupby(['grade_label','answer_language'],sort=True)]
    take=[]
    while len(take)<n:
        for b in buckets:
            if b and len(take)<n:take.append(b.pop())
    return frame.loc[take].copy()
if len(qa)<CFG['n_test']+CFG['n_dev']+100:raise ValueError('Insufficient reviewed QA for the configured split sizes')
test_groups=select_groups(qa,CFG['n_test'],CFG['seed'])
rest=qa[~qa.group_id.isin(test_groups)]
dev_groups=select_groups(rest,CFG['n_dev'],CFG['seed']+1)
qa['split']='train';qa.loc[qa.group_id.isin(test_groups),'split']='test_pool';qa.loc[qa.group_id.isin(dev_groups),'split']='dev_pool'
test=balanced_take(qa[qa.split=='test_pool'],CFG['n_test'],CFG['seed']).sort_values('qa_id')
dev=balanced_take(qa[qa.split=='dev_pool'],CFG['n_dev'],CFG['seed']).sort_values('qa_id')
qa.loc[qa.qa_id.isin(test.qa_id),'split']='test';qa.loc[qa.qa_id.isin(dev.qa_id),'split']='dev'
train=qa[qa.split=='train'].copy();test=qa[qa.split=='test'].copy();dev=qa[qa.split=='dev'].copy()
assert not set(train.group_id)&(set(dev.group_id)|set(test.group_id))
assert not set(dev.group_id)&set(test.group_id)
assert not set(train.chunk_id)&set(test.chunk_id)
if len(train)<80 or len(test)!=CFG['n_test'] or len(dev)!=CFG['n_dev']:
    raise ValueError('Grouped split too small: inspect grouping/QA coverage before changing the prespecified design')
TRAIN_ALLOWED=~corpus.group_id.isin(set(test.group_id)|set(dev.group_id))
# Unselected questions from test/dev groups are explicitly withheld, never recycled into training.
CORPUS_KEY=digest([source_hashes['corpus'],corpus[['chunk_id','grade_min','grade_max','text']].to_dict('records')])[:20]
SPLIT_KEY=digest(qa[['qa_id','split','group_id','question','answer']].to_dict('records'))[:20]
atomic_parquet(DATA/'corpus.parquet',corpus);atomic_parquet(DATA/'qa_partitions.parquet',qa)
pd.DataFrame(near_pairs,columns=['left','right','jaccard']).to_csv(DATA/'near_duplicate_pairs.csv',index=False)
atomic_json(DATA/'provenance.json',dict(source_hashes=source_hashes,corpus_key=CORPUS_KEY,split_key=SPLIT_KEY,
    n_raw=len(raw),n_retained=len(corpus),n_qa=len(qa),n_train=len(train),n_dev=len(dev),n_test=len(test),
    grade_policy='upper_endpoint_band_eligibility',qa_reviewed=bool(QA_REVIEW_CSV),split_claim='exploratory book-family-disjoint'))
qa[['qa_id','question','answer','source_text','grade_label','answer_language','review_status','answer_exact_in_source']].assign(
    decision='',corrected_question='',corrected_answer='').to_csv(DATA/'qa_review_template.csv',index=False)
log('data',f'{len(corpus):,} passages; {len(train)} train / {len(dev)} dev / {len(test)} test. Withheld unused questions stay outside training.')
print(pd.crosstab(qa.grade_label,qa.split).to_string())
missing_training_grades=sorted(set(test.grade_label)-set(train.grade_label))
if missing_training_grades:log('data warning','No supervised training QA for these test bands; report as held-out-grade behavior, not fully covered training.',bands=missing_training_grades)
stage_state('data','complete',study=str(STUDY))


[    0.7 min |  644.4 min left] data: 37,708 passages; 483 train / 100 dev / 200 test. Withheld unused questions stay outside training.
split        dev  dev_pool  test  test_pool  train
grade_label                                       
1              0         0    17          2      0
11-12          9         4    16          0      0
2              0         0    19          0      0
3             12         0    21          0     40
4             12         0    21          0     42
5             15         0    23          0     36
6             15         0    20          0    105
7             12         0    19          0    103
8             15         1    20          0     96
9-10          10         0    24          0     61
[    0.7 min |  644.4 min left] data warning: No supervised training QA for these test bands; report as held-out-grade behavior, not fully covered training.


In [5]:
# CELL 5 — publication exports and statistical helpers (used after every relevant stage)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'savefig.dpi':300,'pdf.fonttype':42})
def table(name,df):
    df.to_csv(TAB/(name+'.csv'),index=False)
    try:(TAB/(name+'.tex')).write_text(df.to_latex(index=False,float_format=lambda x:f'{x:.4f}',escape=True),encoding='utf-8')
    except Exception as exc:log('export','LaTeX export unavailable',table=name,error=str(exc))
    return df

def savefig(name,fig):
    fig.tight_layout()
    for ext in ['pdf','png']:fig.savefig(FIG/(name+'.'+ext),bbox_inches='tight')
    plt.close(fig)

def cluster_ci(values,groups,n_boot=None,seed=42):
    frame=pd.DataFrame({'value':values,'group':groups}).dropna()
    if not len(frame):return [None,None,None,0,0]
    sums=frame.groupby('group').value.agg(['sum','count']);g=len(sums)
    if g<2:return [float(frame.value.mean()),None,None,len(frame),g]
    rng=np.random.default_rng(seed);idx=rng.integers(0,g,size=(n_boot or CFG['bootstrap_replicates'],g))
    means=sums['sum'].to_numpy()[idx].sum(1)/sums['count'].to_numpy()[idx].sum(1)
    lo,hi=np.quantile(means,[.025,.975]);return [float(frame.value.mean()),float(lo),float(hi),len(frame),g]

def holm(ps):
    ps=np.asarray(ps,float);out=np.full(len(ps),np.nan);valid=np.where(np.isfinite(ps))[0];order=valid[np.argsort(ps[valid])];run=0
    for rank,i in enumerate(order):run=max(run,(len(order)-rank)*ps[i]);out[i]=min(1.,run)
    return out

def cluster_signflip(values,groups,reps=9999):
    f=pd.DataFrame({'v':values,'g':groups}).dropna();a=f.groupby('g').v.sum().to_numpy()
    if len(a)<2:return np.nan
    rng=np.random.default_rng(42);observed=abs(a.sum());count=0
    for start in range(0,reps,500):
        signs=rng.choice([-1,1],size=(min(500,reps-start),len(a)))
        count+=int((abs(signs@a)>=observed-1e-12).sum())
    return (count+1)/(reps+1)

inventory=corpus.groupby(['grade_label','language'],sort=True).agg(passages=('chunk_id','size'),books=('book_id','nunique')).reset_index()
table('table01_corpus_inventory',inventory)
table('table02_split_coverage',qa.groupby(['split','grade_label','answer_language']).agg(questions=('qa_id','size'),passages=('chunk_id','nunique'),groups=('group_id','nunique')).reset_index())
fig,ax=plt.subplots(figsize=(9,3));ax.axis('off')
steps=['Grouped QA split','BM25 / dense / hybrid','Grade policy + reranking','Protected chat prompt','Answer + human audit']
for i,s in enumerate(steps):
    ax.text(i,0,s,ha='center',va='center',wrap=True,bbox=dict(boxstyle='round,pad=.5',fc='#e9f2fb',ec='#3a6ea5'),fontsize=9)
    if i<4:ax.annotate('',(i+.62,0),(i+.38,0),arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-.65,4.7);ax.set_ylim(-.6,.6);savefig('figure01_pipeline',fig)


## Embeddings, indices and per-question retrieval

Embedding shards are committed separately. Reusing a stage requires the corpus order, encoder revision, preprocessing and query identities to match. E5 uses its required `query:`/`passage:` prefixes. FAISS runs on CPU; GPU memory is reserved for model inference. Search and metric records retain chunk positions plus the immutable corpus manifest.


In [6]:
# CELL 6 — sharded dense embeddings and a saved sparse BM25 index
import faiss
faiss.omp_set_num_threads(max(1,min(4,os.cpu_count() or 1)))
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
ACTIVE=qa[qa.split.isin(['train','dev','test'])].sort_values('qa_id').reset_index(drop=True)
QPOS=dict(zip(ACTIVE.qa_id,ACTIVE.index));QLOOK=ACTIVE.set_index('qa_id',drop=False)
RETRIEVAL=STUDY/'retrieval';RETRIEVAL.mkdir(exist_ok=True)
DENSE={};RETS={}

def dense_artifacts(tag):
    repo=EMBEDDERS[tag];rev=revision(repo)
    key=digest([CORPUS_KEY,repo,rev,CFG['embedding_max_tokens'],CFG['embedding_shard'],'normalized_mean_v2',tag=='e5'])[:20]
    folder=ROOT/'embeddings'/key;folder.mkdir(parents=True,exist_ok=True)
    texts=corpus.text.tolist();n=len(texts);size=CFG['embedding_shard']
    qkey=digest([key,ACTIVE[['qa_id','question']].to_dict('records')])[:20]
    qfile=folder/f'queries_{qkey}.npy';model=None;times=[]
    try:
        todo=[(a,min(a+size,n)) for a in range(0,n,size) if not (folder/f'shard_{a:06d}.npy').exists()]
        if todo or not qfile.exists():
            BUDGET.check(600)
            model=SentenceTransformer(repo,revision=rev,token=HF_TOKEN,device=DEVICE,trust_remote_code=False)
            model.max_seq_length=CFG['embedding_max_tokens'];model.half()
        def encode(batch,prefix):
            bs=CFG['embedding_batch']
            while True:
                try:return model.encode([prefix+x for x in batch],batch_size=bs,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False).astype('float32')
                except torch.cuda.OutOfMemoryError:
                    cleanup();bs//=2
                    if bs<1:raise
        for j,(start,end) in enumerate(todo):
            BUDGET.check(max(60,max(times[-3:],default=0)*1.5));t=time.monotonic()
            arr=encode(texts[start:end],'passage: ' if tag=='e5' else '')
            if not np.isfinite(arr).all():raise ValueError('Nonfinite embeddings')
            atomic_npy(folder/f'shard_{start:06d}.npy',arr);times.append(time.monotonic()-t)
            log('embedding',f'{tag}: {end}/{n} passages checkpointed; ETA {np.mean(times[-5:])*(len(todo)-j-1)/60:.1f} min')
        if not qfile.exists():
            BUDGET.check(120);qe=encode(ACTIVE.question.tolist(),'query: ' if tag=='e5' else '');atomic_npy(qfile,qe)
        embs=np.concatenate([np.load(folder/f'shard_{a:06d}.npy') for a in range(0,n,size)])
        if len(embs)!=n:raise ValueError('Embedding row count mismatch')
        ipath=folder/'index.faiss'
        if not ipath.exists():
            idx=faiss.IndexFlatIP(embs.shape[1]);idx.add(embs);tmp=folder/'index.tmp';faiss.write_index(idx,str(tmp));os.replace(tmp,ipath)
        else:idx=faiss.read_index(str(ipath))
        assert idx.ntotal==n
        qe=np.load(qfile);scores,ids=idx.search(qe,CFG['search_depth'])
        artifact_key=digest([key,qkey,CFG['search_depth']])[:20]
        out=RETRIEVAL/f'{tag}_{artifact_key}.parquet'
        atomic_parquet(out,pd.DataFrame({'qa_id':ACTIVE.qa_id,'ids':list(ids),'scores':list(scores)}))
        atomic_json(folder/'manifest.json',dict(corpus_key=CORPUS_KEY,repo=repo,revision=rev,rows=n,dimension=embs.shape[1],preprocessing=key))
        return {'embeddings':embs,'queries':qe,'index':idx,'key':artifact_key}, {'ids':ids,'scores':scores}
    finally:
        if model is not None:del model
        cleanup()

for tag in ['bge_m3']+(['e5'] if CFG['run_e5'] else []):
    try:
        if tag=='e5':BUDGET.check(30*60)
        DENSE[tag],RETS[tag]=dense_artifacts(tag);stage_state('embedding_'+tag,'complete')
    except BudgetStop as exc:log(tag,str(exc));stage_state('embedding_'+tag,'pending');break
    except Exception as exc:
        log(tag,'FAILED: no substitute encoder is used',error=str(exc));stage_state('embedding_'+tag,'failed',error=str(exc))
        if tag=='bge_m3':raise
if 'bge_m3' not in RETS:raise BudgetStop('Primary embeddings incomplete. Saved shards will resume.')

bmkey=digest([CORPUS_KEY,'bm25_unicode_lower_v2',1.5,.75,.25])[:20]
bmdir=ROOT/'indices'/bmkey;bmdir.mkdir(parents=True,exist_ok=True)
if (bmdir/'complete.json').exists():
    BM=sparse.load_npz(bmdir/'weights.npz');vocabulary=read_json(bmdir/'vocabulary.json')
else:
    log('bm25','Building sparse term/document weights')
    docs=[tokens(t) for t in corpus.text];bm=BM25Okapi(docs);vocabulary={t:i for i,t in enumerate(bm.idf)}
    rows=[];cols=[];vals=[]
    for j,tf in enumerate(bm.doc_freqs):
        k=bm.k1*(1-bm.b+bm.b*bm.doc_len[j]/bm.avgdl)
        for term,freq in tf.items():
            rows.append(vocabulary[term]);cols.append(j);vals.append(bm.idf[term]*freq*(bm.k1+1)/(freq+k))
    BM=sparse.csr_matrix((np.asarray(vals,dtype='float32'),(rows,cols)),shape=(len(vocabulary),len(corpus)))
    sparse.save_npz(bmdir/'weights.tmp.npz',BM);os.replace(bmdir/'weights.tmp.npz',bmdir/'weights.npz')
    atomic_json(bmdir/'vocabulary.json',vocabulary);atomic_json(bmdir/'complete.json',{'key':bmkey})
    del docs,bm,rows,cols,vals;gc.collect()
bids=[];bscores=[]
for question in ACTIVE.question:
    tt=[vocabulary[t] for t in tokens(question) if t in vocabulary]
    scores=np.asarray(BM[tt].sum(0)).ravel() if tt else np.zeros(len(corpus))
    ids=np.argsort(-scores,kind='stable')[:CFG['search_depth']];bids.append(ids);bscores.append(scores[ids])
RETS['bm25']={'ids':np.asarray(bids),'scores':np.asarray(bscores)}
atomic_parquet(RETRIEVAL/f'bm25_{bmkey}_{SPLIT_KEY}.parquet',pd.DataFrame({'qa_id':ACTIVE.qa_id,'ids':bids,'scores':bscores}))
del BM;gc.collect()
hr=[];hs=[]
for a,b in zip(RETS['bge_m3']['ids'],RETS['bm25']['ids']):
    score=defaultdict(float)
    for ranks in [a,b]:
        for rank,doc in enumerate(ranks):score[int(doc)]+=1/(60+rank+1)
    ordered=sorted(score,key=lambda d:(-score[d],d))[:CFG['search_depth']]
    hr.append(ordered);hs.append([score[x] for x in ordered])
RETS['hybrid']={'ids':np.asarray(hr),'scores':np.asarray(hs)}
atomic_parquet(RETRIEVAL/f'hybrid_{DENSE["bge_m3"]["key"]}_{bmkey}_{SPLIT_KEY}.parquet',pd.DataFrame({'qa_id':ACTIVE.qa_id,'ids':hr,'scores':hs}))
log('retrieval','Dense, BM25 and RRF ranked IDs/scores saved.')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

[    8.0 min |  637.0 min left] embedding: e5: 512/37708 passages checkpointed; ETA 5.3 min
[    8.1 min |  636.9 min left] embedding: e5: 1024/37708 passages checkpointed; ETA 5.5 min
[    8.1 min |  636.9 min left] embedding: e5: 1536/37708 passages checkpointed; ETA 5.4 min
[    8.2 min |  636.8 min left] embedding: e5: 2048/37708 passages checkpointed; ETA 5.3 min
[    8.3 min |  636.7 min left] embedding: e5: 2560/37708 passages checkpointed; ETA 5.2 min
[    8.4 min |  636.6 min left] embedding: e5: 3072/37708 passages checkpointed; ETA 5.3 min
[    8.5 min |  636.5 min left] embedding: e5: 3584/37708 passages checkpointed; ETA 5.4 min
[    8.6 min |  636.5 min left] embedding: e5: 4096/37708 passages checkpointed; ETA 5.6 min
[    8.6 min |  636.4 min left] embedding: e5: 4608/37708 passages checkpointed; ETA 5.4 min
[    8.7 min |  636.3 min left] embedding: e5: 5120/37708 passages checkpointed; ETA 5.4 min
[    8.8 min |  636.2 min left] embedding: e5: 5632/37708 passages chec

In [7]:
# CELL 7 — strict search, backfill and development-only soft penalties
GMAX=corpus.grade_max.to_numpy();METHODS={};ALPHA_ROWS=[];RET_ROWS=[]
EVAL_ROWS=ACTIVE[ACTIVE.split.isin(['dev','test'])]

def retrieval_values(ids,row,k=5):
    ids=list(map(int,ids[:k]));grades=GMAX[ids] if ids else np.array([])
    hit=[i+1 for i,d in enumerate(ids) if d==int(row.gold_pos)]
    return {'recall':float(bool(hit)),'mrr':1/hit[0] if hit else 0.,
            'ndcg':1/np.log2(hit[0]+1) if hit else 0.,
            'gcr':float(np.mean(grades>row.grade_max)) if len(ids) else np.nan,
            'grade_distance':float(np.mean(abs(grades-row.grade_max))) if len(ids) else np.nan,
            'returned':len(ids),'full_k':len(ids)==k}

def register_method(name,mapping,description):
    METHODS[name]={'ids':mapping,'description':description}
    rec=[]
    for r in EVAL_ROWS.itertuples():
        ids=mapping[r.qa_id]
        rec.append(dict(qa_id=r.qa_id,split=r.split,group_id=r.group_id,method=name,ids=list(map(int,ids)),
                        **retrieval_values(ids,r,CFG['final_k'])))
    key=digest([name,description,rec])[:20];METHODS[name]['key']=key
    atomic_parquet(RETRIEVAL/f'method_{name}_{key}.parquet',pd.DataFrame(rec));RET_ROWS.extend(rec)

for tag,item in RETS.items():
    register_method(tag,{r.qa_id:item['ids'][QPOS[r.qa_id]][:CFG['final_k']].tolist() for r in EVAL_ROWS.itertuples()},'Unconstrained '+tag)
# Global eligible-grade search, not truncation of a global candidate list.
BM_STRICT=sparse.load_npz(bmdir/'weights.npz')
for tag in ['bge_m3','hybrid']:
    strict={}
    for ceiling in sorted(EVAL_ROWS.grade_max.unique()):
        eligible=np.flatnonzero(GMAX<=ceiling)
        subidx=faiss.IndexFlatIP(DENSE['bge_m3']['embeddings'].shape[1]);subidx.add(DENSE['bge_m3']['embeddings'][eligible])
        part=EVAL_ROWS[EVAL_ROWS.grade_max==ceiling];_,found=subidx.search(DENSE['bge_m3']['queries'][[QPOS[q] for q in part.qa_id]],min(CFG['search_depth'],len(eligible)))
        for row,dd in zip(part.itertuples(),found):
            dense_ids=eligible[dd].tolist()
            if tag=='bge_m3':ids=dense_ids
            else:
                # Compute BM25 over the eligible corpus exactly; do not approximate strict sparse search with a shortlist.
                terms=[vocabulary[t] for t in tokens(row.question) if t in vocabulary]
                sc=np.asarray(BM_STRICT[terms].sum(0)).ravel() if terms else np.zeros(len(corpus))
                sparse_ids=eligible[np.argsort(-sc[eligible],kind='stable')[:CFG['search_depth']]].tolist()
                ff=defaultdict(float)
                for ranklist in [dense_ids,sparse_ids]:
                    for rank,d in enumerate(ranklist):ff[d]+=1/(61+rank)
                ids=sorted(ff,key=lambda d:(-ff[d],d))
            strict[row.qa_id]=ids[:CFG['final_k']]
        del subidx
    register_method(tag+'_strict',strict,'Global band-eligible search, no backfill')
    backfill={};soft={};base=RETS[tag]
    def soft_ids(row,alpha):
        qi=QPOS[row.qa_id];ids=base['ids'][qi][:CFG['candidate_k']];scores=base['scores'][qi][:CFG['candidate_k']]
        rel=(scores-scores.min())/max(1e-9,float(scores.max()-scores.min()))
        adj=rel-alpha*np.maximum(0,GMAX[ids]-row.grade_max)
        return ids[np.argsort(-adj,kind='stable')[:CFG['final_k']]].tolist()
    for row in EVAL_ROWS.itertuples():
        pool=base['ids'][QPOS[row.qa_id]][:CFG['candidate_k']].tolist();keep=[d for d in pool if GMAX[d]<=row.grade_max]
        backfill[row.qa_id]=(keep+[d for d in pool if d not in keep])[:CFG['final_k']]
    register_method(tag+'_backfill',backfill,'Historical top-candidate filter with explicitly allowed backfill')
    options=[]
    for alpha in CFG['alpha_grid']:
        for split in ['dev','test']:
            vals=[retrieval_values(soft_ids(r,alpha),r,CFG['final_k']) for r in EVAL_ROWS[EVAL_ROWS.split==split].itertuples()]
            summary=pd.DataFrame(vals).mean(numeric_only=True).to_dict()
            ALPHA_ROWS.append(dict(retriever=tag,split=split,alpha=alpha,**summary))
            if split=='dev':options.append((summary['recall']-CFG['utility_gcr_weight']*summary['gcr'], -alpha,alpha))
    chosen=max(options)[2]
    register_method(tag+'_soft',{r.qa_id:soft_ids(r,chosen) for r in EVAL_ROWS.itertuples()},f'Dev-selected alpha={chosen}; objective recall-{CFG["utility_gcr_weight"]}*GCR')
    log('tuning',f'{tag}: alpha={chosen} selected on development set only.')
del BM_STRICT;gc.collect()
register_method('oracle',{r.qa_id:[int(r.gold_pos)] for r in EVAL_ROWS.itertuples()},'Single gold evidence-access control; not an assured quality upper bound')

DEPTH_ROWS=[]
for tag,arr in RETS.items():
    for r in EVAL_ROWS.itertuples():
        for k in [1,3,5,10,20,60,120]:
            DEPTH_ROWS.append(dict(method=tag,k=k,qa_id=r.qa_id,split=r.split,group_id=r.group_id,**retrieval_values(arr['ids'][QPOS[r.qa_id]],r,k)))
atomic_parquet(RETRIEVAL/'depth_metrics.parquet',pd.DataFrame(DEPTH_ROWS))
table('table03_retrieval_depth',pd.DataFrame(DEPTH_ROWS).groupby(['split','method','k'])[['recall','mrr','ndcg','gcr','grade_distance','returned']].mean().reset_index())
table('table05_alpha_sensitivity',pd.DataFrame(ALPHA_ROWS))

def export_retrieval():
    frame=pd.DataFrame(RET_ROWS);atomic_parquet(RETRIEVAL/'all_method_metrics.parquet',frame)
    rows=[]
    for (split,name),g in frame.groupby(['split','method']):
        for metric in ['recall','gcr','grade_distance','returned']:
            mean,lo,hi,n,ng=cluster_ci(g[metric],g.group_id)
            rows.append(dict(split=split,method=name,metric=metric,mean=mean,ci_low=lo,ci_high=hi,n=n,groups=ng))
    table('table04_grade_mechanisms',pd.DataFrame(rows))
    contrasts=[];f=frame[frame.split=='test']
    for method in ['bge_m3_strict','bge_m3_soft',f"explicit_grade_s{CFG['primary_seed']}"]:
        if method not in set(f.method):continue
        for metric in ['recall','gcr']:
            p=f.pivot(index='qa_id',columns='method',values=metric)[['bge_m3',method]].dropna()
            delta=p[method]-p.bge_m3;groups=QLOOK.loc[p.index,'group_id'].to_numpy()
            mean,lo,hi,n,ng=cluster_ci(delta.to_numpy(),groups)
            contrasts.append(dict(method=method,metric=metric,n=n,groups=ng,delta=mean,ci_low=lo,ci_high=hi,p_cluster=cluster_signflip(delta.to_numpy(),groups)))
    if contrasts:
        ct=pd.DataFrame(contrasts);ct['p_holm']=holm(ct.p_cluster) if len(ct)==6 else np.nan
        ct['family_complete']=len(ct)==6;table('table_retrieval_paired',ct)
    s=frame[frame.split=='test'].groupby('method')[['recall','gcr']].mean()
    fig,ax=plt.subplots(figsize=(8,5))
    for name,r in s.iterrows():ax.scatter(r.gcr,r.recall);ax.annotate(name,(r.gcr,r.recall),fontsize=7,xytext=(4,3),textcoords='offset points')
    ax.set(xlabel='Above-band passage fraction (lower is better)',ylabel='Gold passage Recall@5');savefig('figure02_retrieval_tradeoff',fig)
    fig,ax=plt.subplots(figsize=(7,4))
    for (tag,split),g in pd.DataFrame(ALPHA_ROWS).groupby(['retriever','split']):
        ax.plot(g.alpha,g.gcr,marker='o',label=f'{tag}, {split}',linestyle='-' if split=='dev' else '--')
    ax.set(xlabel='Penalty alpha',ylabel='Above-band fraction');ax.legend(fontsize=8);savefig('figure03_alpha_sensitivity',fig)
export_retrieval();stage_state('retrieval','complete')


[   14.4 min |  630.6 min left] tuning: bge_m3: alpha=0.8 selected on development set only.
[   14.4 min |  630.6 min left] tuning: hybrid: alpha=0.8 selected on development set only.


## Rerankers: strong relevance control and explicit grade supervision

The pretrained multilingual BGE reranker is the **relevance-only control**. The smaller trained model has two outputs: relevance and metadata eligibility. Its ranking utility is their product. Gold passages receive relevance target 1; other candidates use pretrained-reranker soft labels unless actual training-only relevance judgments are supplied. This is **weakly supervised distillation**, not independently annotated relevance or pedagogical suitability. Random/ungold passages are never declared irrelevant solely because they lack the gold ID.

Training negatives exclude all development/test book groups. Counterfactual grade labels supervise *metadata eligibility only*. Grade-input and no-grade-input variants share the same data and model capacity. Checkpoints retain optimizer, schedule and RNG states during unfinished training; completed runs retain the selected model and delete the bulky optimizer snapshot. To extend an unfinished run, keep its configuration unchanged; new seeds/variants reuse embeddings and teacher scores.


In [8]:
# CELL 8 — resumable relevance scoring, with no fallback system
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
PAIR_RECORDS=[];TEACHER_READY=False
PAIR_DIR=STUDY/'rerankers';PAIR_DIR.mkdir(exist_ok=True)

def ce_inputs(tok,rows,variant):
    left=[];right=[]
    for r in rows:
        head=f"Passage curriculum upper grade: {int(r['passage_grade'])}. "
        if variant=='explicit_grade':head=f"Requested curriculum upper grade: {int(r['target_grade'])}. "+head
        left.append(head+'Question: '+r['question']);right.append(r['passage'])
    return tok(left,right,padding=True,truncation='only_second',max_length=CFG['ce_max_tokens'],return_tensors='pt')

def score_relevance(model,tok,question,ids):
    outputs=[];bs=12
    for a in range(0,len(ids),bs):
        BUDGET.check(30);ch=ids[a:a+bs]
        enc=tok([question]*len(ch),corpus.iloc[ch].text.tolist(),padding=True,truncation='only_second',max_length=CFG['ce_max_tokens'],return_tensors='pt').to(DEVICE)
        with torch.inference_mode():val=torch.sigmoid(model(**enc).logits.view(-1).float()).cpu().tolist()
        outputs.extend(val)
    return outputs

def teacher_stage():
    repo=RELEVANCE_MODEL;rev=revision(repo)
    key=digest([SPLIT_KEY,CORPUS_KEY,DENSE['bge_m3']['key'],repo,rev,CFG['ce_max_tokens'],CFG['ce_candidates_train'],CFG['ce_candidates_eval']])[:20]
    folder=PAIR_DIR/('teacher_'+key);folder.mkdir(exist_ok=True)
    result=[];model=tok=None;allowed=np.flatnonzero(TRAIN_ALLOWED.to_numpy());dur=[]
    try:
        for j,r in enumerate(ACTIVE.itertuples()):
            fp=folder/(r.qa_id+'.json')
            if fp.exists():result.extend(read_json(fp));continue
            BUDGET.check(120)
            if model is None:
                tok=AutoTokenizer.from_pretrained(repo,revision=rev,token=HF_TOKEN)
                model=AutoModelForSequenceClassification.from_pretrained(repo,revision=rev,token=HF_TOKEN,
                    torch_dtype=torch.float16,attn_implementation='eager').to(DEVICE).eval()
            t=time.monotonic();ids=RETS['bge_m3']['ids'][QPOS[r.qa_id]].tolist()
            if r.split=='train':
                ids=[int(r.gold_pos)]+[d for d in ids if TRAIN_ALLOWED.iloc[d] and d!=r.gold_pos]
                ids=ids[:CFG['ce_candidates_train']]
                if len(ids)<CFG['ce_candidates_train']:
                    rng=np.random.default_rng(int(r.qa_id[:8],16))
                    for d in rng.permutation(allowed):
                        if int(d) not in ids:ids.append(int(d))
                        if len(ids)>=CFG['ce_candidates_train']:break
                assert all(TRAIN_ALLOWED.iloc[d] for d in ids)
            else:ids=ids[:CFG['ce_candidates_eval']]
            scores=score_relevance(model,tok,r.question,ids)
            records=[dict(qa_id=r.qa_id,split=r.split,doc=int(d),teacher=float(s)) for d,s in zip(ids,scores)]
            atomic_json(fp,records);result.extend(records);dur.append(time.monotonic()-t)
            if j%10==0:log('relevance reranker',f'{j+1}/{len(ACTIVE)} questions saved; uncached ETA upper estimate {np.mean(dur[-10:])*(len(ACTIVE)-j-1)/60:.1f} min')
        return result,key
    finally:
        if model is not None:del model
        cleanup()

if CFG['run_rerankers']:
    try:
        PAIR_RECORDS,TEACHER_KEY=teacher_stage();TEACHER_READY=True
        pairs=pd.DataFrame(PAIR_RECORDS);atomic_parquet(PAIR_DIR/'teacher_pairs.parquet',pairs)
        ranking={qid:g.sort_values(['teacher','doc'],ascending=[False,True]).doc.head(CFG['final_k']).tolist()
                 for qid,g in pairs[pairs.split.isin(['dev','test'])].groupby('qa_id')}
        register_method('relevance_ce',ranking,'Pretrained multilingual relevance-only cross-encoder');export_retrieval()
        stage_state('relevance_ce','complete')
    except BudgetStop as exc:log('relevance_ce',str(exc));stage_state('relevance_ce','pending')
    except Exception as exc:log('relevance_ce','Failed; other systems remain usable',error=str(exc));stage_state('relevance_ce','failed',error=str(exc))
else:stage_state('relevance_ce','disabled')


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

[   14.7 min |  630.3 min left] relevance reranker: 1/783 questions saved; uncached ETA upper estimate 28.4 min
[   14.8 min |  630.2 min left] relevance reranker: 11/783 questions saved; uncached ETA upper estimate 10.4 min
[   14.9 min |  630.1 min left] relevance reranker: 21/783 questions saved; uncached ETA upper estimate 5.5 min
[   14.9 min |  630.0 min left] relevance reranker: 31/783 questions saved; uncached ETA upper estimate 5.4 min
[   15.0 min |  630.0 min left] relevance reranker: 41/783 questions saved; uncached ETA upper estimate 6.6 min
[   15.2 min |  629.9 min left] relevance reranker: 51/783 questions saved; uncached ETA upper estimate 7.9 min
[   15.3 min |  629.7 min left] relevance reranker: 61/783 questions saved; uncached ETA upper estimate 9.1 min
[   15.4 min |  629.6 min left] relevance reranker: 71/783 questions saved; uncached ETA upper estimate 9.9 min
[   15.5 min |  629.5 min left] relevance reranker: 81/783 questions saved; uncached ETA upper estimate

In [9]:
# CELL 9 — resumable student training, held-out model selection and seed ablations
from torch.utils.data import DataLoader
CE_MODELS={};CE_HISTORY=[]

def make_training_rows():
    judged={}
    if RELEVANCE_CSV:
        jf=pd.read_csv(RELEVANCE_CSV)
        if not set(jf.qa_id).issubset(set(train.qa_id)):raise ValueError('Relevance file contains dev/test/unknown queries; only training labels are accepted here')
        if jf.duplicated(['qa_id','chunk_id']).any() or not jf.relevance.between(0,1).all():raise ValueError('Invalid relevance judgments')
        judged={(r.qa_id,str(r.chunk_id)):float(r.relevance) for r in jf.itertuples()}
    rows=[];ceilings=sorted(corpus.grade_max.unique());rng=np.random.default_rng(CFG['seed'])
    for p in PAIR_RECORDS:
        if p['split']!='train':continue
        q=QLOOK.loc[p['qa_id']];doc=corpus.iloc[p['doc']]
        rel=judged.get((q.qa_id,str(doc.chunk_id)),1. if p['doc']==q.gold_pos else p['teacher'])
        other=[int(g) for g in ceilings if g!=q.grade_max]
        for target in [int(q.grade_max),int(rng.choice(other))]:
            rows.append(dict(qa_id=q.qa_id,doc=int(doc.pos),question=q.question,passage=doc.text,
                target_grade=target,passage_grade=int(doc.grade_max),rel_label=rel,eligible_label=float(doc.grade_max<=target),
                supervision='judged' if (q.qa_id,str(doc.chunk_id)) in judged else 'gold' if doc.pos==q.gold_pos else 'teacher_soft'))
    return rows

def student_scores(model,tok,variant,partition):
    records=[];block=[]
    def flush():
        if not block:return
        BUDGET.check(30);enc=ce_inputs(tok,block,variant).to(DEVICE)
        with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):v=torch.sigmoid(model(**enc).logits.float()).cpu().numpy()
        for row,score in zip(block,v):records.append(dict(qa_id=row['qa_id'],doc=row['doc'],relevance=float(score[0]),eligibility=float(score[1]),utility=float(score.prod())))
        block.clear()
    for p in PAIR_RECORDS:
        if p['split']!=partition:continue
        q=QLOOK.loc[p['qa_id']];d=corpus.iloc[p['doc']]
        block.append(dict(qa_id=q.qa_id,doc=int(d.pos),question=q.question,passage=d.text,target_grade=q.grade_max,passage_grade=d.grade_max))
        if len(block)>=48:flush()
    flush();return pd.DataFrame(records)

def ranking_objective(scores):
    vals=[]
    for qid,g in scores.groupby('qa_id'):
        ids=g.sort_values(['utility','doc'],ascending=[False,True]).doc.head(CFG['final_k']).tolist();vals.append(retrieval_values(ids,QLOOK.loc[qid],CFG['final_k']))
    out=pd.DataFrame(vals).mean(numeric_only=True).to_dict()
    return out['recall']-CFG['utility_gcr_weight']*out['gcr'],out

def rng_state():
    npstate=np.random.get_state()
    return dict(python=random.getstate(),numpy=[npstate[0],npstate[1].tolist(),npstate[2],npstate[3],npstate[4]],
                torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all())
def restore_rng(state):
    random.setstate(state['python']);a=state['numpy'];np.random.set_state((a[0],np.asarray(a[1],dtype='uint32'),a[2],a[3],a[4]))
    torch.set_rng_state(state['torch']);torch.cuda.set_rng_state_all(state['cuda'])

def train_student(variant,seed,rows):
    repo=CFG['ce_base'];rev=revision(repo)
    key=digest([SPLIT_KEY,TEACHER_KEY,repo,rev,variant,seed,CFG['ce_epochs'],CFG['ce_batch'],CFG['ce_max_tokens'],CFG['ce_lr'],
                file_hash(RELEVANCE_CSV) if RELEVANCE_CSV else None,'two_head_product_v2'])[:20]
    folder=PAIR_DIR/f'{variant}_s{seed}_{key}';folder.mkdir(exist_ok=True);last=folder/'last_training.pt';bestdir=folder/'best'
    if (folder/'complete.json').exists():return folder
    BUDGET.check(180);seed_all(seed)
    tok=AutoTokenizer.from_pretrained(repo,revision=rev,token=HF_TOKEN)
    model=AutoModelForSequenceClassification.from_pretrained(repo,revision=rev,token=HF_TOKEN,num_labels=2).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['ce_lr'],weight_decay=.01)
    nb=math.ceil(len(rows)/CFG['ce_batch']);sched=get_linear_schedule_with_warmup(opt,int(.1*nb*CFG['ce_epochs']),nb*CFG['ce_epochs'])
    scaler=torch.amp.GradScaler('cuda');epoch=0;next_batch=0;best=-float('inf');history=[]
    def checkpoint():
        tmp=last.with_suffix('.tmp')
        torch.save(dict(model=model.state_dict(),optimizer=opt.state_dict(),scheduler=sched.state_dict(),scaler=scaler.state_dict(),
                        epoch=epoch,next_batch=next_batch,best=best,history=history,rng=rng_state()),tmp)
        os.replace(tmp,last)
    try:
        if last.exists():
            state=torch.load(last,map_location='cpu',weights_only=True)
            model.load_state_dict(state['model']);opt.load_state_dict(state['optimizer']);sched.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            epoch=state['epoch'];next_batch=state['next_batch'];best=state['best'];history=state['history'];restore_rng(state['rng']);del state
            log('training',f'Resumed {variant} seed {seed}, epoch {epoch+1}, next batch {next_batch}')
        while epoch<CFG['ce_epochs']:
            generator=torch.Generator().manual_seed(seed+epoch)
            loader=DataLoader(rows,batch_size=CFG['ce_batch'],shuffle=True,generator=generator,collate_fn=lambda x:x,num_workers=0)
            model.train();losses=[];epoch_start=time.monotonic()
            for step,batch in enumerate(loader):
                if step<next_batch:continue
                BUDGET.check(120)
                enc=ce_inputs(tok,batch,variant).to(DEVICE)
                labels=torch.tensor([[r['rel_label'],r['eligible_label']] for r in batch],dtype=torch.float32,device=DEVICE)
                opt.zero_grad(set_to_none=True)
                with torch.autocast('cuda',dtype=torch.float16):loss=torch.nn.functional.binary_cross_entropy_with_logits(model(**enc).logits.float(),labels)
                if not torch.isfinite(loss):raise ValueError('Nonfinite reranker loss')
                scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(opt);scaler.update();sched.step();next_batch=step+1;losses.append(float(loss.detach()))
                if next_batch%CFG['ce_checkpoint_steps']==0:
                    checkpoint();log('training',f'{variant} seed {seed} epoch {epoch+1}: {next_batch}/{nb}; checkpoint saved; epoch elapsed {(time.monotonic()-epoch_start)/60:.1f} min')
            checkpoint();model.eval();scores=student_scores(model,tok,variant,'dev');value,metrics=ranking_objective(scores)
            history.append(dict(epoch=epoch+1,loss=float(np.mean(losses)) if losses else None,objective=value,**metrics))
            if value>best or not bestdir.exists():
                best=value;temp=folder/'best_pending'
                if temp.exists():shutil.rmtree(temp)
                model.save_pretrained(temp,safe_serialization=True);tok.save_pretrained(temp)
                if bestdir.exists():shutil.rmtree(bestdir)
                os.replace(temp,bestdir)
                atomic_parquet(folder/'selected_dev_scores.parquet',scores)
            epoch+=1;next_batch=0;checkpoint();atomic_json(folder/'history.json',history)
            log('training',f'{variant} seed {seed}: epoch {epoch} selected-dev objective={best:.4f}')
        atomic_json(folder/'complete.json',dict(key=key,variant=variant,seed=seed,best_dev_objective=best,supervision='weak+judgments' if RELEVANCE_CSV else 'weak'))
        last.unlink(missing_ok=True)  # Only unfinished optimizer snapshots are necessary for interruption recovery.
        return folder
    except BaseException:
        checkpoint();raise
    finally:
        del model,opt,sched,scaler
        cleanup()

if TEACHER_READY:
    train_rows=make_training_rows();atomic_parquet(PAIR_DIR/'training_pairs.parquet',pd.DataFrame(train_rows).drop(columns=['passage']))
    stop=False
    # Complete the primary explicit-grade model before extra seeds/ablations.
    schedule=[(v,s) for s in CFG['ce_seeds'] for v in CFG['ce_variants']]
    for variant,seed in schedule:
        tag=f'{variant}_s{seed}'
        try:
            if seed!=CFG['primary_seed'] or variant!='explicit_grade':BUDGET.check(6*3600)
            folder=train_student(variant,seed,train_rows);CE_MODELS[tag]=str(folder)
            model=tok=None
            try:
                scorefile=folder/'heldout_scores.parquet'
                if scorefile.exists():sc=pd.read_parquet(scorefile)
                else:
                    BUDGET.check(180);tok=AutoTokenizer.from_pretrained(folder/'best')
                    model=AutoModelForSequenceClassification.from_pretrained(folder/'best',torch_dtype=torch.float16).to(DEVICE).eval()
                    sc=pd.concat([student_scores(model,tok,variant,p).assign(split=p) for p in ['dev','test']],ignore_index=True)
                    atomic_parquet(scorefile,sc)
                rank={qid:g.sort_values(['utility','doc'],ascending=[False,True]).doc.head(CFG['final_k']).tolist() for qid,g in sc.groupby('qa_id')}
                register_method(tag,rank,'Two-head student; '+variant+f'; seed={seed}; development-selected epoch')
                CE_HISTORY.extend([dict(variant=variant,seed=seed,**x) for x in read_json(folder/'history.json',[])])
                stage_state(tag,'complete')
            finally:
                if model is not None:del model
                cleanup()
        except BudgetStop as exc:log(tag,str(exc));stage_state(tag,'pending');break
        except Exception as exc:log(tag,'Failed; no soft-reranking substitution',error=str(exc));stage_state(tag,'failed',error=str(exc));cleanup()
    if CE_HISTORY:table('table_reranker_training_history',pd.DataFrame(CE_HISTORY))
    export_retrieval()


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

The tokenizer you are loading from '/kaggle/working/nctb_grade_audit_v2/studies/828b7e9ff0551d817215/rerankers/no_grade_input_s43_1409c31a5f6466b7f272/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/Multilingual-MiniLM-L12-H384 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[   58.2 min |  586.8 min left] training: explicit_grade seed 44 epoch 1: 50/644; checkpoint saved; epoch elapsed 0.2 min
[   58.4 min |  586.6 min left] training: explicit_grade seed 44 epoch 1: 100/644; checkpoint saved; epoch elapsed 0.3 min
[   58.6 min |  586.4 min left] training: explicit_grade seed 44 epoch 1: 150/644; checkpoint saved; epoch elapsed 0.5 min
[   58.8 min |  586.2 min left] training: explicit_grade seed 44 epoch 1: 200/644; checkpoint saved; epoch elapsed 0.7 min
[   59.0 min |  586.0 min left] training: explicit_grade seed 44 epoch 1: 250/644; checkpoint saved; epoch elapsed 0.9 min
[   59.1 min |  585.9 min left] training: explicit_grade seed 44 epoch 1: 300/644; checkpoint saved; epoch elapsed 1.0 min
[   59.3 min |  585.7 min left] training: explicit_grade seed 44 epoch 1: 350/644; checkpoint saved; epoch elapsed 1.2 min
[   59.5 min |  585.5 min left] training: explicit_grade seed 44 epoch 1: 400/644; checkpoint saved; epoch elapsed 1.4 min
[   59.6 min |  5

The tokenizer you are loading from '/kaggle/working/nctb_grade_audit_v2/studies/828b7e9ff0551d817215/rerankers/explicit_grade_s44_f580f9afc61eac6ee064/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/Multilingual-MiniLM-L12-H384 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[   67.4 min |  577.6 min left] training: no_grade_input seed 44 epoch 1: 50/644; checkpoint saved; epoch elapsed 0.2 min
[   67.6 min |  577.4 min left] training: no_grade_input seed 44 epoch 1: 100/644; checkpoint saved; epoch elapsed 0.3 min
[   67.8 min |  577.2 min left] training: no_grade_input seed 44 epoch 1: 150/644; checkpoint saved; epoch elapsed 0.5 min
[   67.9 min |  577.1 min left] training: no_grade_input seed 44 epoch 1: 200/644; checkpoint saved; epoch elapsed 0.7 min
[   68.1 min |  576.9 min left] training: no_grade_input seed 44 epoch 1: 250/644; checkpoint saved; epoch elapsed 0.9 min
[   68.3 min |  576.7 min left] training: no_grade_input seed 44 epoch 1: 300/644; checkpoint saved; epoch elapsed 1.0 min
[   68.5 min |  576.5 min left] training: no_grade_input seed 44 epoch 1: 350/644; checkpoint saved; epoch elapsed 1.2 min
[   68.6 min |  576.4 min left] training: no_grade_input seed 44 epoch 1: 400/644; checkpoint saved; epoch elapsed 1.4 min
[   68.8 min |  5

The tokenizer you are loading from '/kaggle/working/nctb_grade_audit_v2/studies/828b7e9ff0551d817215/rerankers/no_grade_input_s44_974dab2b16fa6e6ba1a5/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [10]:
# CELL 10 — held-out counterfactual grade response, not a training-pool curve
curve=[];tag=f"explicit_grade_s{CFG['primary_seed']}"
if tag in CE_MODELS:
    model=None
    try:
        folder=Path(CE_MODELS[tag]);fp=folder/'counterfactual_grade_curve.parquet'
        if fp.exists():curve_df=pd.read_parquet(fp)
        else:
            BUDGET.check(180);tok=AutoTokenizer.from_pretrained(folder/'best');model=AutoModelForSequenceClassification.from_pretrained(folder/'best',torch_dtype=torch.float16).to(DEVICE).eval()
            for r in test.sort_values('qa_id').head(40).itertuples():
                d=corpus.iloc[RETS['bge_m3']['ids'][QPOS[r.qa_id]][0]]
                rows=[dict(question=r.question,passage=d.text,target_grade=int(g),passage_grade=int(d.grade_max)) for g in sorted(corpus.grade_max.unique())]
                with torch.inference_mode():out=torch.sigmoid(model(**ce_inputs(tok,rows,'explicit_grade').to(DEVICE)).logits.float()).cpu().numpy()
                for row,v in zip(rows,out):curve.append(dict(qa_id=r.qa_id,group_id=r.group_id,target_grade=row['target_grade'],passage_grade=d.grade_max,relevance=v[0],eligibility=v[1],utility=v.prod()))
            curve_df=pd.DataFrame(curve);atomic_parquet(fp,curve_df)
        table('table_counterfactual_grade_response',curve_df)
        fig,ax=plt.subplots(figsize=(7,4))
        for metric in ['relevance','eligibility','utility']:
            g=curve_df.groupby('target_grade')[metric].mean();ax.plot(g.index,g,marker='o',label=metric)
        ax.set(ylim=(0,1),xlabel='Requested upper grade (same question/passage)',ylabel='Mean model score');ax.legend();savefig('figure04_counterfactual_grade_response',fig)
    except BudgetStop:stage_state('counterfactual_curve','pending')
    except Exception as exc:log('counterfactual_curve','Failed',error=str(exc))
    finally:
        if model is not None:del model
        cleanup()


The tokenizer you are loading from '/kaggle/working/nctb_grade_audit_v2/studies/828b7e9ff0551d817215/rerankers/explicit_grade_s42_31da0dfc9bd4e49e0dfd/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [11]:
# CELL 11 — independent lexical fitting and held-out diagnostics
LEX_DIR=STUDY/'lexical';LEX_DIR.mkdir(exist_ok=True)
lexfile=LEX_DIR/'training_vocabulary.parquet'
if lexfile.exists():lex=pd.read_parquet(lexfile)
else:
    vocab={};frequencies=Counter()
    for r in corpus[TRAIN_ALLOWED & corpus.language.isin(['bn','en'])].itertuples():
        for t in set(tokens(r.text)):
            key=(r.language,t);frequencies[key]+=1
            old=vocab.get(key,(99,99));vocab[key]=(min(old[0],int(r.grade_min)),min(old[1],int(r.grade_max)))
    lex=pd.DataFrame([dict(language=l,token=t,first_lower=g[0],first_upper=g[1],document_frequency=frequencies[(l,t)]) for (l,t),g in vocab.items()])
    atomic_parquet(lexfile,lex)
VOCABS={freq:{(r.language,r.token):int(r.first_upper) for r in lex[lex.document_frequency>=freq].itertuples()} for freq in [1,3]}

def lexical_score(text,grade,language,min_frequency=1):
    tt=tokens(text);v=VOCABS[min_frequency];dist=[max(0,v[(language,t)]-grade) for t in tt if (language,t) in v]
    return {'gas':1/(1+float(np.mean(dist))) if dist else None,
            'coverage':len(dist)/len(tt) if tt else 0.,'known_tokens':len(dist),'total_tokens':len(tt)}

from scipy.stats import spearmanr
held=corpus[~TRAIN_ALLOWED & corpus.language.isin(['bn','en'])].sample(min(1200,int((~TRAIN_ALLOWED & corpus.language.isin(['bn','en'])).sum())),random_state=42)
validation=[]
for r in held.itertuples():
    for freq in [1,3]:
        x=lexical_score(r.text,5,r.language,freq)
        validation.append(dict(chunk_id=r.chunk_id,group_id=r.group_id,language=r.language,source_upper_grade=r.grade_max,
            frequency_threshold=freq,fixed_target=5,length=len(tokens(r.text)),**x))
validation=pd.DataFrame(validation);atomic_parquet(LEX_DIR/'heldout_diagnostics.parquet',validation)
lexrows=[]
for (language,freq),g in validation.groupby(['language','frequency_threshold']):
    v=g.dropna(subset=['gas'])
    rho,p=spearmanr(v.source_upper_grade,v.gas) if len(v)>2 and v.gas.nunique()>1 else (np.nan,np.nan)
    lexrows.append(dict(language=language,min_document_frequency=freq,n=len(g),scorable=len(v),mean_coverage=g.coverage.mean(),
                        grade_gas_rho=rho,p_descriptive=p,validation='held_out_book_diagnostic_not_teacher_validity'))
table('table06_lexical_diagnostics',pd.DataFrame(lexrows))
fig,ax=plt.subplots(figsize=(7,4))
for lang,g in validation[validation.frequency_threshold==1].groupby('language'):
    s=g.groupby('source_upper_grade').coverage.mean();ax.plot(s.index,s,marker='o',label=lang)
ax.set(xlabel='Held-out source upper grade',ylabel='Recognized vocabulary fraction',ylim=(0,1));ax.legend();savefig('figure05_heldout_vocabulary_coverage',fig)
stage_state('lexical','complete')


## Generation and evaluation

The primary comparisons share the same grade-conditioned instruction. `dense_plain`/`strict_plain` form the prompt ablation. Passage text is packed under a token budget **without truncating the assembled prompt**. The full question must survive a decode check. No reference answer is used to select evidence. Reference substring survival is logged only afterwards as a diagnostic, not as a correctness measure.

Generation is greedy NF4, float16 computation, one GPU, batch size 4 (automatic smaller-batch OOM retry). Native Phi support is used with `trust_remote_code=False` to avoid the historical remote-code cache mismatch; each model must pass a bilingual smoke test. Optional families are replications, not promises of completion.

Every example is atomically saved with status, exact input IDs, delivered prompt/evidence, model revision, output, token counts and timing. Retryable failures do not become empty successful answers. Evaluation/figures refresh after each complete condition and partial progress is clearly labeled.


In [12]:
# CELL 12 — scoring, generation coverage, book-cluster CIs, paired comparisons
from sacrebleu.metrics import CHRF
from rouge_score import rouge_scorer
CHRF_METRIC=CHRF(char_order=6,word_order=0,beta=2)
class UnicodeTokenizer:
    def tokenize(self,text):return tokens(text)
ROUGE=rouge_scorer.RougeScorer(['rougeL'],tokenizer=UnicodeTokenizer(),use_stemmer=False)
GEN=STUDY/'generations';GEN.mkdir(exist_ok=True)
RUN_PATHS={}

def answer_metrics(pred,ref,grade,language):
    a,b=Counter(tokens(pred)),Counter(tokens(ref));same=sum((a&b).values())
    precision=same/max(1,sum(a.values()));recall=same/max(1,sum(b.values()))
    x=lexical_score(pred,grade,language);x3=lexical_score(pred,grade,language,3)
    digits=str.maketrans('০১২৩৪৫৬৭৮৯','0123456789')
    numerical_ref=bool(re.fullmatch(r'[\d\s.,%+−/-]+',str(ref).translate(digits).strip()))
    return dict(chrf=CHRF_METRIC.sentence_score(pred,[ref]).score/100,
        rougeL=ROUGE.score(ref,pred)['rougeL'].fmeasure,token_f1=2*precision*recall/(precision+recall) if precision+recall else 0.,
        normalized_exact=float(norm(pred)==norm(ref)),numeric_exact=float(pred.translate(digits).strip()==ref.translate(digits).strip()) if numerical_ref else None,
        gas=x['gas'],coverage=x['coverage'],gas_minfreq3=x3['gas'],coverage_minfreq3=x3['coverage'],
        cmr=float(x['gas']<CFG['cmr_threshold']) if x['gas'] is not None else None,
        script_language_match=float(script_language(pred)==language),answer_tokens_unicode=sum(a.values()))

def all_generation_records():
    records=[]
    for (model,system),folder in RUN_PATHS.items():
        for p in (folder/'records').glob('*.json'):records.append(read_json(p))
    return records

def refresh_outputs():
    rec=all_generation_records()
    flat=[];coverage=[]
    for (model,system),folder in RUN_PATHS.items():
        rr=[read_json(p) for p in (folder/'records').glob('*.json')];success=sum(r['status']=='success' for r in rr)
        coverage.append(dict(model=model,system=system,expected=len(test),attempted=len(rr),successful=success,
            failed=sum(r['status']=='failed' for r in rr),pending=len(test)-success,complete=success==len(test),run_key=folder.name))
        for r in rr:
            base={k:r.get(k) for k in ['qa_id','model','system','status','grade_label','grade_max','answer_language','group_id','prediction','reference','attempt',
                'input_tokens','output_tokens','amortized_batch_seconds','packing_seconds','retrieval_lookup_seconds','batch_size','stop_reason','gold_chunk_delivered','reference_substring_delivered']}
            if r['status']=='success':base.update(answer_metrics(r['prediction'],r['reference'],r['grade_max'],r['answer_language']))
            flat.append(base)
    known={(r['model'],r['system']) for r in coverage}
    for modeltag in CFG['generation_models']:
        planned=list(SYSTEM_SPECS) if modeltag==CFG['generation_models'][0] else REPLICATION_SYSTEMS
        for system in planned:
            if (modeltag,system) not in known:
                coverage.append(dict(model=modeltag,system=system,expected=len(test),attempted=0,successful=0,failed=0,pending=len(test),complete=False,run_key=None))
    cov=pd.DataFrame(coverage);table('table07_execution_coverage',cov)
    if not flat:return
    df=pd.DataFrame(flat);atomic_parquet(STUDY/'scores.parquet',df)
    good=df[df.status=='success'].copy();summ=[]
    if good.empty:return
    for (model,system),g in good.groupby(['model','system']):
        for metric in ['chrf','rougeL','token_f1','gas','coverage','cmr','script_language_match']:
            mean,lo,hi,n,ng=cluster_ci(g[metric],g.group_id)
            summ.append(dict(model=model,system=system,metric=metric,mean=mean,ci_low=lo,ci_high=hi,n_scorable=n,groups=ng,
                successful=len(g),expected=len(test),complete=len(g)==len(test)))
    table('table08_generation_metrics',pd.DataFrame(summ))
    # Only complete matched cells enter inferential comparisons. All are exploratory on this QA pool.
    stat=[];primary=['strict_grade','soft_grade','grade_ce_grade'];metrics=['chrf','gas']
    for model in good.model.unique():
        dg=good[good.model==model];complete=set(cov[(cov.model==model)&cov.complete].system)
        if 'dense_grade' not in complete:continue
        for system in primary:
            if system not in complete:continue
            for metric in metrics:
                pivot=dg.pivot(index='qa_id',columns='system',values=metric)[['dense_grade',system]].dropna()
                diff=pivot[system]-pivot.dense_grade;groups=QLOOK.loc[pivot.index,'group_id'].to_numpy()
                mean,lo,hi,n,ng=cluster_ci(diff.to_numpy(),groups)
                rawp=cluster_signflip(diff.to_numpy(),groups)
                stat.append(dict(model=model,system=system,baseline='dense_grade',metric=metric,n=n,groups=ng,
                    difference=mean,ci_low=lo,ci_high=hi,p_cluster_signflip=rawp,interpretation='exploratory; symmetric cluster sign-flip null'))
    stats=pd.DataFrame(stat)
    if not stats.empty:
        # Correct only when the entire prespecified six-test family for a model is available.
        stats['p_holm']=np.nan;stats['family_complete']=False
        for model,g in stats.groupby('model'):
            if len(g)==6:
                stats.loc[g.index,'p_holm']=holm(g.p_cluster_signflip);stats.loc[g.index,'family_complete']=True
        table('table09_paired_comparisons',stats)
        fig,ax=plt.subplots(figsize=(8,4));g=stats[stats.metric=='chrf'].reset_index(drop=True)
        for i,r in g.iterrows():
            if pd.notna(r.ci_low):ax.errorbar(r.difference,i,xerr=[[r.difference-r.ci_low],[r.ci_high-r.difference]],fmt='o')
        ax.set_yticks(range(len(g)),g.model+' / '+g.system);ax.axvline(0,color='grey',linestyle='--');ax.set_xlabel('Paired chrF difference; book-group bootstrap 95% CI');savefig('figure06_paired_effects',fig)
    per=[]
    for (model,system,lang,grade),g in good.groupby(['model','system','answer_language','grade_label']):
        per.append(dict(model=model,system=system,language=lang,grade_band=grade,n=len(g),chrf=g.chrf.mean(),gas=g.gas.mean(),coverage=g.coverage.mean()))
    per=pd.DataFrame(per);table('table_generation_strata',per)
    if 'qwen2.5-7b' in set(good.model):
        fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
        for ax,lang in zip(axes,['bn','en']):
            for system in ['dense_grade','strict_grade','soft_grade','grade_ce_grade']:
                g=per[(per.model=='qwen2.5-7b')&(per.system==system)&(per.language==lang)].copy()
                g['order']=g.grade_band.str.split('-').str[0].astype(int);g=g.sort_values('order')
                ax.plot(g.order,g.chrf,marker='o',label=system)
            ax.set(title=lang,xlabel='Target grade / lower endpoint of band',ylabel='Mean chrF');ax.legend(fontsize=7)
        savefig('figure07_language_grade_results',fig)
    runtime=good.groupby(['model','system']).agg(outputs=('qa_id','size'),mean_batch_amortized_seconds=('amortized_batch_seconds','mean'),
        mean_input_tokens=('input_tokens','mean'),mean_output_tokens=('output_tokens','mean')).reset_index()
    runtime['scope']='generation batches only; excludes model loading and precomputed retrieval'
    table('table10_generation_runtime',runtime)
    fig,ax=plt.subplots(figsize=(8,4))
    for (model,system),g in good.groupby(['model','system']):
        ax.scatter(g.amortized_batch_seconds.mean(),g.chrf.mean());ax.annotate(model+'/'+system,(g.amortized_batch_seconds.mean(),g.chrf.mean()),fontsize=6)
    ax.set(xlabel='Generation batch seconds / output (not online end-to-end)',ylabel='Mean chrF');savefig('figure08_generation_cost',fig)


In [13]:
# CELL 13 — protected prompt packing, native templates, smoke tests, atomic per-item generation
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
SYSTEM_SPECS={
 'dense_grade':('bge_m3',True), 'strict_grade':('bge_m3_strict',True),
 'dense_plain':('bge_m3',False), 'strict_plain':('bge_m3_strict',False),
 'soft_grade':('bge_m3_soft',True),'relevance_ce_grade':('relevance_ce',True),
 'grade_ce_grade':(f"explicit_grade_s{CFG['primary_seed']}",True),
 'hybrid_grade':('hybrid',True),'bm25_grade':('bm25',True),
 'closed_book_grade':(None,True),'oracle_grade':('oracle',True),
}
REPLICATION_SYSTEMS=['dense_grade','strict_grade','soft_grade','grade_ce_grade']

def pack_prompt(tok,row,ids,grade_prompt,retrieval=True):
    question=str(row.question);language=row.answer_language
    if language=='bn':
        instruction='বাংলায় সংক্ষিপ্ত ও সরাসরি উত্তর দাও।'
        if grade_prompt:instruction+=f' লক্ষ্য শ্রেণি বা শ্রেণি-পরিসর {row.grade_label}। এই স্তরের উপযোগী সহজ ভাষা ব্যবহার করো।'
        if retrieval:instruction+=' শুধু নিচের প্রমাণ ব্যবহার করো। প্রমাণে উত্তর না থাকলে বলো: "প্রদত্ত প্রমাণ থেকে উত্তর দেওয়া যায় না।"'
        qprefix='প্রশ্ন: ';eprefix='প্রমাণ:';suffix='উত্তর:'
    else:
        instruction='Answer concisely and directly in English.'
        if grade_prompt:instruction+=f' Use language suitable for curriculum grade or band {row.grade_label}.'
        if retrieval:instruction+=' Use only the evidence below. If it does not support an answer, say: "The provided evidence is insufficient to answer."'
        qprefix='Question: ';eprefix='Evidence:';suffix='Answer:'
    protected=instruction+'\n\n'+qprefix+question
    def render(parts):
        context='\n\n'.join(f'[{i+1}] '+text for i,text in enumerate(parts))
        content=protected+('\n\n'+eprefix+'\n'+context if retrieval else '')+'\n\n'+suffix
        msgs=[{'role':'user','content':content}]
        ids_=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True)
        return ids_,msgs,context
    base,_,_=render([])
    if len(base)>CFG['gen_input_tokens']-32:raise ValueError('Question/instructions exceed safe input budget; do not truncate the question')
    passage_tokens=[tok.encode(str(corpus.iloc[d].text),add_special_tokens=False) for d in ids]
    cap=CFG['gen_passage_tokens'] if len(ids)>1 else CFG['gen_input_tokens']-len(base)-16
    parts=[tok.decode(x[:cap],skip_special_tokens=True).rstrip('�') for x in passage_tokens]
    delivered,msgs,context=render(parts)
    # Trim evidence only; protected question/instructions and chat control tokens are never truncated.
    while len(delivered)>CFG['gen_input_tokens']:
        lengths=[len(tok.encode(x,add_special_tokens=False)) for x in parts]
        j=int(np.argmax(lengths));old=tok.encode(parts[j],add_special_tokens=False)
        cut=min(len(old),max(8,len(delivered)-CFG['gen_input_tokens']))
        parts[j]=tok.decode(old[:-cut],skip_special_tokens=True).rstrip('�') if cut<len(old) else ''
        delivered,msgs,context=render(parts)
    decoded=tok.decode(delivered,skip_special_tokens=False)
    if unicodedata.normalize('NFC',question) not in unicodedata.normalize('NFC',decoded):raise AssertionError('Question failed round-trip token survival check')
    evidence=[{'doc':int(d),'chunk_id':str(corpus.iloc[d].chunk_id),'text':text,'grade_min':int(corpus.iloc[d].grade_min),'grade_max':int(corpus.iloc[d].grade_max)} for d,text in zip(ids,parts)]
    return {'input_ids':delivered,'messages':msgs,'delivered_prompt':decoded,'delivered_evidence':evidence,'context':context}

@torch.inference_mode()
def decode_batch(model,tok,packed):
    data=tok.pad({'input_ids':[p['input_ids'] for p in packed]},padding=True,return_tensors='pt').to(DEVICE)
    assert data.input_ids.shape[1]<=CFG['gen_input_tokens']
    torch.cuda.synchronize();t=time.monotonic()
    output=model.generate(**data,max_new_tokens=CFG['gen_output_tokens'],do_sample=False,num_beams=1,
        pad_token_id=tok.pad_token_id,eos_token_id=model.generation_config.eos_token_id,use_cache=True)
    torch.cuda.synchronize();elapsed=time.monotonic()-t
    generated=output[:,data.input_ids.shape[1]:].cpu().tolist();results=[]
    eos=model.generation_config.eos_token_id;ends=set(eos if isinstance(eos,list) else [eos])
    for ids in generated:
        stop=next((i for i,x in enumerate(ids) if x in ends),len(ids));actual=ids[:stop]
        text=tok.decode(actual,skip_special_tokens=True).strip()
        results.append(dict(prediction=text,output_ids=actual,output_tokens=len(actual),stop_reason='eos' if stop<len(ids) else 'length'))
    return results,elapsed


def generation_folder(modeltag,system,rev):
    method,grade=SYSTEM_SPECS[system]
    key=digest([SPLIT_KEY,modeltag,rev,system,METHODS[method]['key'] if method else 'closed_book',
        'protected_bilingual_chat_v2',CFG['gen_input_tokens'],CFG['gen_output_tokens'],CFG['gen_passage_tokens'],'NF4_fp16_double'])[:20]
    folder=GEN/modeltag/system/key;(folder/'records').mkdir(parents=True,exist_ok=True)
    atomic_json(folder/'manifest.json',dict(key=key,model=modeltag,revision=rev,system=system,method=method,grade_prompt=grade,expected=test.qa_id.tolist(),
        input_cap=CFG['gen_input_tokens'],output_cap=CFG['gen_output_tokens'],corpus_key=CORPUS_KEY,split_key=SPLIT_KEY))
    RUN_PATHS[(modeltag,system)]=folder
    return folder

def failure_record(row,modeltag,system,attempt,error):
    return dict(qa_id=row.qa_id,model=modeltag,system=system,status='failed',attempt=attempt,error=str(error),
        grade_label=row.grade_label,grade_max=int(row.grade_max),answer_language=row.answer_language,group_id=row.group_id,
        prediction=None,reference=row.answer)

def run_system(model,tok,modeltag,system,folder):
    method,grade=SYSTEM_SPECS[system];pending=[]
    for r in test.sort_values('qa_id').itertuples():
        old=read_json(folder/'records'/(r.qa_id+'.json'),{})
        if old.get('status')=='success':continue
        if old.get('attempt',0)>=CFG['max_attempts']:continue
        pending.append((r,old.get('attempt',0)+1))
    batch_size=CFG['gen_batch'];recent=[];cursor=0;committed=0
    while cursor<len(pending):
        BUDGET.check(max(120,3*max(recent[-10:],default=30)))
        selected=pending[cursor:cursor+batch_size];packs=[];rows=[];metas=[]
        for row,attempt in selected:
            try:
                t=time.monotonic();ids=METHODS[method]['ids'][row.qa_id] if method else [];lookup=time.monotonic()-t
                t=time.monotonic();pack=pack_prompt(tok,row,ids,grade,retrieval=method is not None);packing=time.monotonic()-t
                packs.append(pack);rows.append((row,attempt));metas.append((ids,lookup,packing))
            except Exception as exc:atomic_json(folder/'records'/(row.qa_id+'.json'),failure_record(row,modeltag,system,attempt,exc))
        if not packs:cursor+=len(selected);continue
        try:
            answers,elapsed=decode_batch(model,tok,packs);recent.append(elapsed)
        except torch.cuda.OutOfMemoryError:
            cleanup()
            if batch_size>1:batch_size=max(1,batch_size//2);log('generation',f'{modeltag}: reduced batch size to {batch_size} after OOM');continue
            for row,attempt in rows:atomic_json(folder/'records'/(row.qa_id+'.json'),failure_record(row,modeltag,system,attempt,'OOM at batch size 1'))
            cursor+=len(selected);continue
        except Exception as exc:
            for row,attempt in rows:atomic_json(folder/'records'/(row.qa_id+'.json'),failure_record(row,modeltag,system,attempt,exc))
            raise RuntimeError('Model generation failed; remaining work is pending, not empty predictions') from exc
        for (row,attempt),pack,ans,(ids,lookup,packing) in zip(rows,packs,answers,metas):
            if not ans['prediction']:
                atomic_json(folder/'records'/(row.qa_id+'.json'),failure_record(row,modeltag,system,attempt,'Empty decoded answer'));continue
            record=dict(qa_id=row.qa_id,model=modeltag,system=system,status='success',attempt=attempt,
                grade_label=row.grade_label,grade_max=int(row.grade_max),answer_language=row.answer_language,group_id=row.group_id,
                question=row.question,reference=row.answer,revision=REVISIONS[MODEL_REGISTRY[modeltag]],
                retrieved_ids=list(map(int,ids)),input_tokens=len(pack['input_ids']),**ans,**pack,
                amortized_batch_seconds=elapsed/len(rows),batch_size=len(rows),packing_seconds=packing,retrieval_lookup_seconds=lookup,
                gold_chunk_delivered=any(e['doc']==row.gold_pos and bool(e['text']) for e in pack['delivered_evidence']),
                reference_substring_delivered=bool(row.answer) and str(row.answer) in pack['context'])
            atomic_json(folder/'records'/(row.qa_id+'.json'),record);committed+=1
        cursor+=len(selected)
        if cursor%10==0 or cursor==len(pending):
            eta=np.percentile(recent[-20:],90)*math.ceil((len(pending)-cursor)/batch_size)/60
            log('generation',f'{modeltag}/{system}: {cursor}/{len(pending)} pending items processed; ETA {eta:.1f} min; committed outputs survive restart')
    return committed

# Discover all configured cache paths first, so a resumed run can score without reloading models.
for modeltag in CFG['generation_models']:
    try:
        rev=revision(MODEL_REGISTRY[modeltag])
        systems=list(SYSTEM_SPECS) if modeltag==CFG['generation_models'][0] else REPLICATION_SYSTEMS
        for system in systems:
            method,_=SYSTEM_SPECS[system]
            if method is None or method in METHODS:generation_folder(modeltag,system,rev)
            else:stage_state(f'gen_{modeltag}_{system}','pending_dependency',missing_method=method)
    except Exception as exc:log('model registry',f'{modeltag} unavailable',error=str(exc));stage_state('model_'+modeltag,'unavailable',error=str(exc))
refresh_outputs()

for modeltag in CFG['generation_models']:
    jobs=[(system,folder) for (mt,system),folder in RUN_PATHS.items() if mt==modeltag]
    todo=[]
    for system,folder in jobs:
        records=[read_json(p) for p in (folder/'records').glob('*.json')]
        if sum(r['status']=='success' for r in records)<len(test):todo.append((system,folder))
    if not todo:continue
    model=tok=None
    try:
        BUDGET.check(12*60);repo=MODEL_REGISTRY[modeltag];rev=revision(repo)
        log('generation',f'Loading {modeltag} in NF4 on {DEVICE}; no automatic CPU offload')
        tok=AutoTokenizer.from_pretrained(repo,revision=rev,token=HF_TOKEN,trust_remote_code=False)
        if not tok.chat_template:raise ValueError('This model has no native chat template; define a documented adapter before using it')
        tok.padding_side='left'
        if tok.pad_token_id is None:tok.pad_token=tok.eos_token
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True)
        model=AutoModelForCausalLM.from_pretrained(repo,revision=rev,token=HF_TOKEN,trust_remote_code=False,
            quantization_config=quant,device_map={'':CFG['generation_device']},torch_dtype=torch.float16,attn_implementation='eager').eval()
        tok.save_pretrained(GEN/modeltag/'tokenizer');model.config.save_pretrained(GEN/modeltag/'model_config')
        # One Bangla and one English DEV smoke test; no test-set-dependent model selection.
        smoke=dev.groupby('answer_language',group_keys=False).head(1)
        if len(smoke)<2:raise ValueError('Bilingual smoke test needs both languages in development')
        ps=[pack_prompt(tok,r,METHODS['bge_m3']['ids'][r.qa_id],True) for r in smoke.itertuples()]
        outputs,seconds=decode_batch(model,tok,ps)
        if any(not r['prediction'] for r in outputs):raise ValueError('Bilingual smoke test produced empty output')
        atomic_json(GEN/modeltag/'smoke_test.json',dict(qa_ids=smoke.qa_id.tolist(),answers=outputs,seconds=seconds,manual_quality_review_required=True))
        log('smoke test',f'{modeltag}: valid nonempty outputs, {seconds/len(ps):.2f} sec/output. Correctness is not certified by this test.')
        for system,folder in todo:
            try:
                run_system(model,tok,modeltag,system,folder)
                n=sum(read_json(p)['status']=='success' for p in (folder/'records').glob('*.json'))
                stage_state(f'gen_{modeltag}_{system}','complete' if n==len(test) else 'incomplete',successful=n,expected=len(test))
                refresh_outputs()
            except BudgetStop:refresh_outputs();raise
    except BudgetStop as exc:log('generation',str(exc));break
    except Exception as exc:log('generation',f'{modeltag} failed smoke test or execution; no quality scores for missing answers',error=str(exc));stage_state('model_'+modeltag,'failed',error=str(exc))
    finally:
        if model is not None:del model
        if tok is not None:del tok
        cleanup()
refresh_outputs()


[   76.6 min |  568.4 min left] generation: Loading qwen2.5-7b in NF4 on cuda:0; no automatic CPU offload


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[   79.0 min |  566.0 min left] smoke test: qwen2.5-7b: valid nonempty outputs, 15.07 sec/output. Correctness is not certified by this test.
[   82.0 min |  563.0 min left] generation: qwen2.5-7b/dense_grade: 20/200 pending items processed; ETA 27.7 min; committed outputs survive restart
[   85.0 min |  560.0 min left] generation: qwen2.5-7b/dense_grade: 40/200 pending items processed; ETA 24.4 min; committed outputs survive restart
[   88.0 min |  557.0 min left] generation: qwen2.5-7b/dense_grade: 60/200 pending items processed; ETA 21.6 min; committed outputs survive restart
[   90.9 min |  554.1 min left] generation: qwen2.5-7b/dense_grade: 80/200 pending items processed; ETA 18.3 min; committed outputs survive restart
[   93.9 min |  551.1 min left] generation: qwen2.5-7b/dense_grade: 100/200 pending items processed; ETA 15.3 min; committed outputs survive restart
[   96.8 min |  548.2 min left] generation: qwen2.5-7b/dense_grade: 120/200 pending items processed; ETA 12.3 min; com

## Teacher evaluation — real ratings only

Inspect and correct `data/qa_review_template.csv` **before a final publishable rerun**. For output evaluation, this section samples the **same questions across four complete systems**, balances grade/language, assigns random blind item IDs, and exports one row per item/rater. The private key contains system/model identity and should not be given to raters.

Raters see the target band, question, reference, response and exact delivered evidence. Rate each dimension 1–5, or leave it blank with a reason. Never fill blank ratings automatically. Record independent ratings before adjudication; document teacher qualifications, consent, compensation and the scope of institutional review appropriate to your study. The notebook cannot obtain teacher participation or certify educational validity.


In [14]:
# CELL 14 — paired blind packet; ingestion; ordinal agreement with bootstrap uncertainty
HUMAN=STUDY/'human';HUMAN.mkdir(exist_ok=True)
DIMS=['correctness','evidence_support','vocabulary_suitability','conceptual_suitability','instructional_usefulness']
RUBRIC={
 'correctness':'1 incorrect/off-topic; 2 mostly incorrect; 3 partly correct; 4 correct with minor omission; 5 fully correct for this question',
 'evidence_support':'1 unsupported/contradicted; 2 mostly unsupported; 3 mixed; 4 mostly supported; 5 all substantive claims supported by DELIVERED evidence',
 'vocabulary_suitability':'1 largely inaccessible; 2 many unexplained advanced terms; 3 mixed; 4 mostly accessible; 5 appropriate vocabulary with necessary explanations',
 'conceptual_suitability':'1 prerequisites far beyond target; 2 substantial missing prerequisites; 3 mixed; 4 minor gap; 5 concepts and explanation fit the target curriculum band',
 'instructional_usefulness':'1 misleading/unusable; 2 little help; 3 some help; 4 helpful; 5 clear, useful and sufficient for the question',
}
atomic_json(HUMAN/'rubric.json',RUBRIC)
records=all_generation_records();primary=CFG['generation_models'][0]
chosen_systems=['dense_grade','strict_grade','soft_grade','grade_ce_grade']
by={(r['system'],r['qa_id']):r for r in records if r['model']==primary and r['status']=='success'}
common=set.intersection(*[{qid for s,qid in by if s==system} for system in chosen_systems]) if chosen_systems else set()
packetfile=HUMAN/'blind_annotation_template.csv';keyfile=HUMAN/'PRIVATE_item_key.csv'
if len(common)==len(test):
    sample=balanced_take(test[test.qa_id.isin(common)],min(CFG['teacher_questions'],len(common)),CFG['seed'])
    jobs=[(qid,s) for qid in sample.qa_id for s in chosen_systems];rng=np.random.default_rng(CFG['seed']);rng.shuffle(jobs)
    packet=[];keys=[]
    for i,(qid,system) in enumerate(jobs):
        r=by[(system,qid)];item='item_'+digest([SPLIT_KEY,primary,qid,system,r['revision'],r['input_ids'],r['prediction'],'blind_v2'])[:12]
        keys.append(dict(item_id=item,qa_id=qid,system=system,model=primary,group_id=r['group_id']))
        for rater in CFG['teacher_raters']:
            packet.append(dict(item_id=item,rater_id=rater,grade_band=r['grade_label'],answer_language=r['answer_language'],question=r['question'],
                reference=r['reference'],response=r['prediction'],delivered_evidence=r['context'],**{d:'' for d in DIMS},notes=''))
    pd.DataFrame(packet).to_csv(packetfile,index=False);pd.DataFrame(keys).to_csv(keyfile,index=False)
    log('human evaluation',f'Exported {len(jobs)} blind outputs / {len(packet)} rating rows. Do not distribute PRIVATE_item_key.csv.')
else:
    (HUMAN/'PENDING.md').write_text(f'Paired packet pending: {len(common)}/{len(test)} questions available across all four required systems. Resume missing generation cells. No synthetic ratings were created.')
    log('human evaluation',f'Packet pending until four matched cells finish ({len(common)}/{len(test)} available).')

if TEACHER_RATINGS_CSV:
    import krippendorff
    ratings=pd.read_csv(TEACHER_RATINGS_CSV)
    if not keyfile.exists():raise ValueError('The matching private item key is required to ingest ratings')
    key=pd.read_csv(keyfile)
    if not set(ratings.item_id).issubset(set(key.item_id)):raise ValueError('Unknown item IDs in returned ratings')
    if ratings.duplicated(['item_id','rater_id']).any():raise ValueError('Duplicate item/rater rows')
    for dim in DIMS:
        ratings[dim]=pd.to_numeric(ratings[dim],errors='raise')
        valid=ratings[dim].dropna()
        if not valid.isin([1,2,3,4,5]).all():raise ValueError('Ratings must be ordinal integers 1–5 or blank')
    if ratings[DIMS].notna().sum().sum()==0:raise ValueError('The returned file contains no actual ratings; leave TEACHER_RATINGS_CSV unset until teachers complete it')
    atomic_parquet(HUMAN/'independent_ratings.parquet',ratings)
    agreement=[]
    for dim in DIMS:
        pivot=ratings.pivot(index='item_id',columns='rater_id',values=dim)
        matrix=pivot.to_numpy().T
        if (pivot.notna().sum(axis=1)>=2).sum()<2:
            agreement.append(dict(dimension=dim,alpha=None,ci_low=None,ci_high=None,items=len(pivot),note='Insufficient overlapping ratings'));continue
        try:a=float(krippendorff.alpha(reliability_data=matrix,level_of_measurement='ordinal',value_domain=[1,2,3,4,5]))
        except ValueError:a=np.nan
        rng=np.random.default_rng(42);boots=[]
        # Cluster on question because each question has several system outputs.
        itemgroups=key.set_index('item_id').loc[pivot.index,'qa_id'];groups=itemgroups.unique();positions=[np.flatnonzero(itemgroups.to_numpy()==g) for g in groups]
        for _ in range(500):
            idx=np.concatenate([positions[j] for j in rng.integers(0,len(groups),len(groups))])
            try:boots.append(krippendorff.alpha(reliability_data=matrix[:,idx],level_of_measurement='ordinal',value_domain=[1,2,3,4,5]))
            except ValueError:pass
        finite=np.array([x for x in boots if np.isfinite(x)]);ci=np.quantile(finite,[.025,.975]) if len(finite)>20 else [np.nan,np.nan]
        agreement.append(dict(dimension=dim,alpha=a,ci_low=ci[0],ci_high=ci[1],items=len(pivot),paired_items=int((pivot.notna().sum(1)>=2).sum()),note='Pre-adjudication ordinal Krippendorff alpha'))
    table('table_human_agreement',pd.DataFrame(agreement))
    merged=ratings.merge(key,on='item_id',validate='many_to_one');human_summ=[]
    for (system,model),g in merged.groupby(['system','model']):
        # Average raters per output before group-bootstrap summaries; do not treat ratings as independent outputs.
        itemmeans=g.groupby(['item_id','group_id'])[DIMS].mean().reset_index()
        for dim in DIMS:
            mean,lo,hi,n,ng=cluster_ci(itemmeans[dim],itemmeans.group_id)
            human_summ.append(dict(system=system,model=model,dimension=dim,mean=mean,ci_low=lo,ci_high=hi,outputs=n,groups=ng))
    h=pd.DataFrame(human_summ);table('table_human_ratings',h)
    fig,ax=plt.subplots(figsize=(8,4));g=h[h.dimension=='conceptual_suitability'].reset_index(drop=True)
    for i,r in g.iterrows():
        if pd.notna(r.ci_low):ax.errorbar(r['mean'],i,xerr=[[r['mean']-r.ci_low],[r.ci_high-r['mean']]],fmt='o')
    ax.set_yticks(range(len(g)),g.system);ax.set(xlim=(1,5),xlabel='Teacher conceptual suitability, book-group bootstrap 95% CI');savefig('figure09_teacher_ratings',fig)
    # Within-grade/language external metric validity, retaining small-cell warnings.
    sf=pd.read_parquet(STUDY/'scores.parquet');hm=merged.groupby(['qa_id','system','model'])[DIMS].mean().reset_index()
    joined=hm.merge(sf,on=['qa_id','system','model'],validate='one_to_one');validrows=[]
    for (grade,lang),g in joined.groupby(['grade_label','answer_language']):
        for dim in ['vocabulary_suitability','conceptual_suitability']:
            vv=g[['gas',dim]].dropna();rho,p=spearmanr(vv.gas,vv[dim]) if len(vv)>=10 and vv.gas.nunique()>1 and vv[dim].nunique()>1 else (np.nan,np.nan)
            validrows.append(dict(grade_band=grade,language=lang,dimension=dim,n=len(vv),spearman=rho,p_descriptive=p,note='Repeated outputs; association is exploratory, not independent-item inference'))
    table('table_external_gas_validity',pd.DataFrame(validrows))
else:
    (FIG/'figure09_PENDING_teacher_ratings.md').write_text('Figure 9 requires genuine returned teacher ratings. Set TEACHER_RATINGS_CSV and rerun the analysis/export cells. No teacher evidence is fabricated.')


[  390.4 min |  254.6 min left] human evaluation: Exported 320 blind outputs / 640 rating rows. Do not distribute PRIVATE_item_key.csv.


In [15]:
# CELL 15 — finish now, not only after every optional experiment. Resume checklist and artifact inventory.
refresh_outputs()
coverage_path=TAB/'table07_execution_coverage.csv'
coverage=pd.read_csv(coverage_path) if coverage_path.exists() else pd.DataFrame()
status=read_json(ROOT/'stage_status.json',{})
manifest=read_json(ROOT/'artifact_manifest.json',{})
manifest.update(last_session=RUN_ID,active_study=DATA_KEY,corpus_key=CORPUS_KEY,split_key=SPLIT_KEY,
    last_updated=datetime.now(timezone.utc).isoformat(),model_revisions=REVISIONS,
    completed_generation_cells=int(coverage.complete.sum()) if len(coverage) else 0,
    generated_files={'tables':sorted(p.name for p in TAB.glob('*.csv')),'figures':sorted(p.name for p in FIG.glob('*.pdf'))})
atomic_json(ROOT/'artifact_manifest.json',manifest)
# Hash final scientific outputs; very large embedding/weight files are identified by their stage manifests.
checksums={str(p.relative_to(ROOT)):file_hash(p) for folder in [DATA,TAB,HUMAN] for p in folder.rglob('*') if p.is_file() and p.suffix in ['.csv','.json','.tex']}
atomic_json(STUDY/'output_checksums.json',checksums)
readme=f"""# NCTB grade audit output
Study: {DATA_KEY}; schema: {CFG['schema']}
Corpus: {CORPUS_KEY}; QA split: {SPLIT_KEY}
This is exploratory evidence on a previously inspected synthetic QA pool.

## Resume
1. Persist this entire nctb_grade_audit_v2 folder as a Kaggle notebook output or dataset.
2. Attach that output and BOTH original inputs to a fresh T4 notebook.
3. Set RESUME_FROM to the attached nctb_grade_audit_v2 folder, or leave automatic discovery on when exactly one exists.
4. Keep data/split/training settings unchanged to resume unfinished work. Add models or seeds to extend the study.
5. Failed items retry only until max_attempts. Increase that bound ONLY after resolving the failure's cause.
6. Metrics can be recalculated from saved answers; adding a scoring metric does not require new generation.

## Interpret correctly
- A completed file is not sufficient: table07 reports expected, successful, failed and pending outputs.
- Missing systems remain missing. There are no fallback results under another system's label.
- GAS/CMR are lexical diagnostics, not validated teacher or learning scores.
- Reranker relevance targets are weak supervision unless actual judged training pairs were supplied.
- Combined grades are bands, not invented exact-grade labels.
- Default timing excludes precomputed retrieval and model loading; no end-to-end speed claim follows.
- No human ratings means no teacher-result figure or educational validation claim.
- Resuming locally saved files requires a durable Kaggle saved version. Abrupt interruption can lose the in-flight batch.

## Files
studies/{DATA_KEY}/data: canonical metadata, grouped QA splits, provenance, review and alignment audits.
embeddings / indices: content-keyed shards, query embeddings and FAISS/BM25 indices.
studies/{DATA_KEY}/retrieval: ranked document IDs/scores, alpha tuning and per-question metrics.
studies/{DATA_KEY}/rerankers: teacher pairs, model checkpoints and held-out scores.
studies/{DATA_KEY}/generations: exact delivered prompts, token IDs, outputs and failure records.
studies/{DATA_KEY}/tables and figures: CSV/LaTeX and 300-dpi PNG/vector PDF exports.
studies/{DATA_KEY}/human: blinded packets, private mapping and real returned ratings if supplied.
logs: per-session environment lock and progress log. stage_status.json: unfinished/failed stages.
"""
(ROOT/'README.md').write_text(readme,encoding='utf-8')
log('finished','Artifacts are ready in '+str(ROOT))
if len(coverage):print(coverage[['model','system','successful','expected','complete']].to_string(index=False))
print('\nSave a durable Kaggle version now. Resume pending blocks from that output in another session.')
print('Tables:',TAB,'\nFigures:',FIG,'\nTeacher packet:',HUMAN)
from IPython.display import display, FileLink
display(FileLink(str(ROOT/'README.md')))
display(FileLink(str(TAB/'table07_execution_coverage.csv'))) if coverage_path.exists() else None


[  390.5 min |  254.5 min left] finished: Artifacts are ready in /kaggle/working/nctb_grade_audit_v2
     model             system  successful  expected  complete
qwen2.5-7b        dense_grade         200       200      True
qwen2.5-7b       strict_grade         200       200      True
qwen2.5-7b        dense_plain         200       200      True
qwen2.5-7b       strict_plain         200       200      True
qwen2.5-7b         soft_grade         200       200      True
qwen2.5-7b relevance_ce_grade         200       200      True
qwen2.5-7b     grade_ce_grade         200       200      True
qwen2.5-7b       hybrid_grade         200       200      True
qwen2.5-7b         bm25_grade         200       200      True
qwen2.5-7b  closed_book_grade         200       200      True
qwen2.5-7b       oracle_grade         200       200      True

Save a durable Kaggle version now. Resume pending blocks from that output in another session.
Tables: /kaggle/working/nctb_grade_audit_v2/studies/828b7e9f

/kaggle/working/nctb_grade_audit_v2/README.md

/kaggle/working/nctb_grade_audit_v2/studies/828b7e9ff0551d817215/tables/table07_execution_coverage.csv

## Optional next sessions and publication checklist

- **Second family:** add `phi3.5-mini`, `llama3.1-8b` or `gemma2-9b` to `CFG['generation_models']`. The first model gets the full matrix; later models get four matched core conditions. Resolve access/compatibility errors before interpreting failure rates. Do not assume completion inside one session.
- **More seeds:** add reranker seeds. Existing seed/model folders remain reusable. Do not pick the seed with the best test score; the primary seed is prespecified.
- **Real labels:** review the QA and book alignment files, then supply their paths. These changes intentionally create a new study/split identity. Teacher ratings alone only require rerunning analysis after restoring the matching study artifacts.
- **New metrics:** extend `answer_metrics` and rerun the scoring/export cells; leave generation signatures unchanged. Avoid treating automated NLI/LLM judges as a teacher replacement.
- **Runtime:** current figures label amortized generation cost honestly. A publishable online benchmark must measure query encoding + search + policy/reranking + packing + generation on the same successful requests, with warmup, GPU synchronization and reported hardware. Cached retrieval lookup is not online retrieval time.
- **Confirmatory evaluation:** commission new independently reviewed questions and a prospectively fixed analysis plan; this notebook does not convert a reused dataset into an unseen benchmark.
- **Optional expensive studies:** broader scaling, precision sweeps, concept/application questions and learner outcomes require separate sessions or external data collection. They are deliberately outside the default 12-hour scope.

### Why no historical “success” claims are carried forward
The old run used whole-prompt truncation, incomplete model coverage and no target-grade input to the learned reranker. This notebook produces a new experimental version. It does not overwrite the old results or mix old and new scores.

### References for implementation choices
- [Qwen model card / native chat formatting](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)
- [Multilingual E5 input prefixes](https://huggingface.co/intfloat/multilingual-e5-large)
- [BGE multilingual reranker](https://huggingface.co/BAAI/bge-reranker-v2-m3)
- [Phi model compatibility guidance](https://huggingface.co/microsoft/Phi-3.5-mini-instruct)

**Validation disclosure:** notebook structure and CPU/data logic were checked locally against the supplied corpus and QA. Prompt-budget tests used the real Qwen tokenizer with an offline chat renderer; native Jinja-template integration remains untested locally. Full CUDA execution, model downloads and measured 12-hour completion require the Kaggle GPU run. Do not describe this notebook as already GPU-benchmarked.
